# Statistical Essentials for Data Scientists
## A Comprehensive Interview Preparation Guide

---

This notebook covers the core statistical concepts tested in rigorous interviews at top product companies (Google, Meta, Netflix, Amazon, Uber, Airbnb, etc.). Each concept is presented with:

* **Rigorous mathematical foundations** — derivations, proofs, and formal definitions
* **Real-world examples** — connecting abstract theory to product/business scenarios
* **Working code** — implementations from scratch and using standard libraries
* **Interview-style insights** — common traps, edge cases, and follow-up questions

### Topics Covered:
1. Probability Foundations & Bayes' Theorem
2. Random Variables, Distributions & Moments
3. Central Limit Theorem & Law of Large Numbers
4. Estimation Theory (MLE, Method of Moments)
5. Confidence Intervals
6. Hypothesis Testing
7. A/B Testing & Experimental Design
8. Bayesian Inference
9. Linear & Logistic Regression
10. Bias-Variance Tradeoff
11. Bootstrap & Resampling Methods
12. Causal Inference
13. Information Theory
14. Markov Chains & Stochastic Processes

# 1. Probability Foundations

---

## 1.1 Axioms of Probability (Kolmogorov)

A probability measure $$P$$ on a sample space $$\Omega$$ satisfies:

1. **Non-negativity**: $$P(A) \geq 0$$ for every event $$A$$
2. **Normalization**: $$P(\Omega) = 1$$
3. **Countable Additivity**: For mutually exclusive events $$A_1, A_2, \ldots$$:

$$P\left(\bigcup_{i=1}^{\infty} A_i\right) = \sum_{i=1}^{\infty} P(A_i)$$

**Key Consequences:**
* $$P(A^c) = 1 - P(A)$$
* $$P(A \cup B) = P(A) + P(B) - P(A \cap B)$$ (Inclusion-Exclusion)
* $$P(\emptyset) = 0$$

---

## 1.2 Conditional Probability

The probability of $$A$$ given $$B$$ has occurred:

$$P(A|B) = \frac{P(A \cap B)}{P(B)}, \quad P(B) > 0$$

**Real-World Example (Product):** At Netflix, if $$A$$ = "user watches a sci-fi movie" and $$B$$ = "user watched Interstellar last week", then $$P(A|B)$$ is the probability of watching sci-fi *conditional on* having watched Interstellar.

### Chain Rule (Multiplication Rule)

$$P(A \cap B) = P(A|B) \cdot P(B) = P(B|A) \cdot P(A)$$

Generalized:

$$P(A_1 \cap A_2 \cap \cdots \cap A_n) = P(A_1) \cdot P(A_2|A_1) \cdot P(A_3|A_1 \cap A_2) \cdots P(A_n|A_1 \cap \cdots \cap A_{n-1})$$

---

## 1.3 Law of Total Probability

If $$B_1, B_2, \ldots, B_n$$ partition $$\Omega$$:

$$P(A) = \sum_{i=1}^{n} P(A|B_i) \cdot P(B_i)$$

**Interview Example:** A ride-sharing company has 3 driver pools: Premium (10%), Standard (60%), Economy (30%). The probability a ride is rated 5-stars given pool:
* $$P(5\star | \text{Premium}) = 0.8$$
* $$P(5\star | \text{Standard}) = 0.5$$  
* $$P(5\star | \text{Economy}) = 0.3$$

Then: $$P(5\star) = 0.8(0.1) + 0.5(0.6) + 0.3(0.3) = 0.08 + 0.30 + 0.09 = 0.47$$

---

## 1.4 Bayes' Theorem

The cornerstone of probabilistic reasoning:

$$P(B_i|A) = \frac{P(A|B_i) \cdot P(B_i)}{P(A)} = \frac{P(A|B_i) \cdot P(B_i)}{\sum_{j=1}^{n} P(A|B_j) \cdot P(B_j)}$$

**Interpretation:**
* $$P(B_i)$$ = **Prior** (belief before seeing evidence)
* $$P(A|B_i)$$ = **Likelihood** (probability of evidence given hypothesis)
* $$P(B_i|A)$$ = **Posterior** (updated belief after seeing evidence)
* $$P(A)$$ = **Evidence/Marginal** (normalizing constant)

### Classic Interview Problem: Disease Testing

A disease affects 1 in 1000 people. A test has:
* Sensitivity (True Positive Rate): $$P(+|\text{Disease}) = 0.99$$
* Specificity (True Negative Rate): $$P(-|\text{Healthy}) = 0.95$$

**Q: If you test positive, what's the probability you actually have the disease?**

$$P(\text{Disease}|+) = \frac{P(+|\text{Disease}) \cdot P(\text{Disease})}{P(+)}$$

$$= \frac{0.99 \times 0.001}{0.99 \times 0.001 + 0.05 \times 0.999} = \frac{0.00099}{0.00099 + 0.04995} \approx 0.0194$$

**Only ~2%!** This is the **base rate fallacy** — when the disease is rare, even a good test produces mostly false positives.

---

## 1.5 Independence

Events $$A$$ and $$B$$ are independent if and only if:

$$P(A \cap B) = P(A) \cdot P(B)$$

Equivalently: $$P(A|B) = P(A)$$ (knowing $$B$$ doesn't change our belief about $$A$$)

**Conditional Independence**: $$A$$ and $$B$$ are conditionally independent given $$C$$ if:

$$P(A \cap B | C) = P(A|C) \cdot P(B|C)$$

> ⚠️ **Interview Trap**: Independence does NOT imply conditional independence, and vice versa!

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# BAYES' THEOREM IN ACTION: Disease Testing
# =============================================================================

def bayes_disease_test(prevalence, sensitivity, specificity):
    """Calculate posterior probability of disease given positive test."""
    # P(+|Disease) * P(Disease)
    p_pos_and_disease = sensitivity * prevalence
    # P(+|Healthy) * P(Healthy)
    p_pos_and_healthy = (1 - specificity) * (1 - prevalence)
    # P(Disease|+)
    p_disease_given_pos = p_pos_and_disease / (p_pos_and_disease + p_pos_and_healthy)
    return p_disease_given_pos

# Demonstrate how posterior changes with prevalence
prevalences = np.linspace(0.0001, 0.5, 1000)
posteriors = [bayes_disease_test(p, sensitivity=0.99, specificity=0.95) for p in prevalences]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Posterior vs Prevalence
axes[0].plot(prevalences, posteriors, 'b-', linewidth=2)
axes[0].axhline(y=0.5, color='r', linestyle='--', alpha=0.7, label='50% threshold')
axes[0].axvline(x=0.001, color='g', linestyle='--', alpha=0.7, label='Prevalence=0.1%')
axes[0].set_xlabel('Disease Prevalence P(Disease)', fontsize=12)
axes[0].set_ylabel('P(Disease | Positive Test)', fontsize=12)
axes[0].set_title('Base Rate Fallacy: Posterior vs Prior', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim([0, 0.5])

# Key insight annotation
axes[0].annotate(f'At 0.1% prevalence:\nP(Disease|+) = {bayes_disease_test(0.001, 0.99, 0.95):.3f}',
                xy=(0.001, bayes_disease_test(0.001, 0.99, 0.95)),
                xytext=(0.05, 0.15),
                fontsize=10, arrowprops=dict(arrowstyle='->', color='green'),
                bbox=dict(boxstyle='round', facecolor='lightyellow'))

# Plot 2: Simulation of 10000 people
np.random.seed(42)
n_people = 10000
prevalence = 0.001

# Simulate true disease status
has_disease = np.random.binomial(1, prevalence, n_people)

# Simulate test results
test_positive = np.zeros(n_people)
for i in range(n_people):
    if has_disease[i]:
        test_positive[i] = np.random.binomial(1, 0.99)  # sensitivity
    else:
        test_positive[i] = np.random.binomial(1, 0.05)  # 1 - specificity

# Count outcomes
true_pos = np.sum((test_positive == 1) & (has_disease == 1))
false_pos = np.sum((test_positive == 1) & (has_disease == 0))
true_neg = np.sum((test_positive == 0) & (has_disease == 0))
false_neg = np.sum((test_positive == 0) & (has_disease == 1))

categories = ['True Positive', 'False Positive', 'True Negative', 'False Negative']
counts = [true_pos, false_pos, true_neg, false_neg]
colors = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12']

axes[1].bar(categories, counts, color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_title(f'Simulation: {n_people:,} People Tested (Prevalence=0.1%)', fontsize=13)
axes[1].set_ylabel('Count', fontsize=12)
for i, (cat, count) in enumerate(zip(categories, counts)):
    axes[1].text(i, count + 5, str(count), ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

empirical_ppv = true_pos / (true_pos + false_pos) if (true_pos + false_pos) > 0 else 0
print(f"\n{'='*60}")
print(f"SIMULATION RESULTS ({n_people:,} people, prevalence = {prevalence*100:.1f}%)")
print(f"{'='*60}")
print(f"True Positives:  {true_pos}")
print(f"False Positives: {false_pos}")
print(f"Empirical PPV:   P(Disease|+) = {true_pos}/{true_pos+false_pos} = {empirical_ppv:.4f}")
print(f"Theoretical PPV: {bayes_disease_test(prevalence, 0.99, 0.95):.4f}")
print(f"\n→ KEY INSIGHT: Even with 99% sensitivity and 95% specificity,")
print(f"  most positive tests are FALSE POSITIVES when disease is rare!")

## 1.6 Classic Interview Probability Problems

### Problem 1: The Monty Hall Problem

You're on a game show with 3 doors. Behind one door is a car; behind the others, goats. You pick Door 1. The host (who knows what's behind each door) opens Door 3, revealing a goat. Should you switch to Door 2?

**Solution using Bayes' Theorem:**

Let $$C_i$$ = "car is behind door $$i$$". Initially $$P(C_1) = P(C_2) = P(C_3) = 1/3$$.

Let $$H_3$$ = "host opens door 3". We need $$P(C_2 | H_3)$$.

$$P(H_3|C_1) = 1/2$$ (host can open either door 2 or 3)

$$P(H_3|C_2) = 1$$ (host MUST open door 3)

$$P(H_3|C_3) = 0$$ (host won't reveal the car)

$$P(C_2|H_3) = \frac{P(H_3|C_2) \cdot P(C_2)}{P(H_3)} = \frac{1 \times 1/3}{1/2 \times 1/3 + 1 \times 1/3 + 0 \times 1/3} = \frac{1/3}{1/2} = \frac{2}{3}$$

**You should ALWAYS switch!** Switching wins with probability $$2/3$$.

---

### Problem 2: Birthday Problem

**Q: How many people do you need in a room for a >50% chance that two share a birthday?**

$$P(\text{no match among } n) = \frac{365}{365} \cdot \frac{364}{365} \cdot \frac{363}{365} \cdots \frac{365-n+1}{365} = \frac{365!}{365^n (365-n)!}$$

$$P(\text{at least one match}) = 1 - \prod_{i=0}^{n-1}\frac{365-i}{365}$$

For $$n = 23$$: $$P \approx 0.507$$ — only 23 people needed!

**Why this matters in tech**: Hash collisions, UUID uniqueness, distributed systems — the birthday paradox shows collisions happen much sooner than intuition suggests. For a hash of $$m$$ bits, expect a collision after $$\approx 2^{m/2}$$ trials.

---

### Problem 3: Coupon Collector's Problem

**Q: A cereal box contains one of $$n$$ different toys. How many boxes must you buy to collect all $$n$$?**

When you have $$i$$ unique toys, the probability of getting a new one is $$\frac{n-i}{n}$$.

Expected boxes for the $$i+1$$-th new toy: $$\frac{n}{n-i}$$

Total expected boxes:

$$E[T] = \sum_{i=0}^{n-1} \frac{n}{n-i} = n \sum_{k=1}^{n} \frac{1}{k} = n \cdot H_n \approx n \ln n + \gamma n$$

where $$H_n$$ is the $$n$$-th harmonic number and $$\gamma \approx 0.5772$$ is the Euler-Mascheroni constant.

**Product application**: If a streaming service has 50 shows and recommends randomly, a user needs $$\approx 50 \ln 50 \approx 225$$ recommendations to see all content.

In [0]:
# =============================================================================
# PROBABILITY SIMULATIONS: Monty Hall, Birthday Problem, Coupon Collector
# =============================================================================

np.random.seed(42)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- MONTY HALL SIMULATION ---
n_sims = 100000
switch_wins = 0
stay_wins = 0

for _ in range(n_sims):
    car = np.random.randint(0, 3)
    choice = np.random.randint(0, 3)
    
    # Host opens a door (not the car, not the player's choice)
    available = [d for d in range(3) if d != choice and d != car]
    host_opens = np.random.choice(available)
    
    # Switch to the remaining door
    switch_to = [d for d in range(3) if d != choice and d != host_opens][0]
    
    if switch_to == car:
        switch_wins += 1
    if choice == car:
        stay_wins += 1

strategies = ['Stay', 'Switch']
win_rates = [stay_wins/n_sims, switch_wins/n_sims]
colors = ['#e74c3c', '#2ecc71']
axes[0].bar(strategies, win_rates, color=colors, edgecolor='black', width=0.5)
axes[0].axhline(y=1/3, color='gray', linestyle='--', alpha=0.7, label='1/3')
axes[0].axhline(y=2/3, color='gray', linestyle='--', alpha=0.7, label='2/3')
axes[0].set_ylim(0, 1)
axes[0].set_title(f'Monty Hall ({n_sims:,} simulations)', fontsize=12)
axes[0].set_ylabel('Win Probability')
for i, v in enumerate(win_rates):
    axes[0].text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')
axes[0].legend()

# --- BIRTHDAY PROBLEM ---
def birthday_probability(n):
    """Exact probability of at least one shared birthday."""
    p_no_match = 1.0
    for i in range(n):
        p_no_match *= (365 - i) / 365
    return 1 - p_no_match

n_values = range(1, 80)
probs = [birthday_probability(n) for n in n_values]

axes[1].plot(n_values, probs, 'b-', linewidth=2)
axes[1].axhline(y=0.5, color='r', linestyle='--', alpha=0.7, label='50% threshold')
axes[1].axvline(x=23, color='g', linestyle='--', alpha=0.7, label='n=23')
axes[1].fill_between(n_values, probs, alpha=0.1)
axes[1].set_xlabel('Number of People')
axes[1].set_ylabel('P(at least one shared birthday)')
axes[1].set_title('Birthday Problem', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# --- COUPON COLLECTOR ---
def coupon_collector_sim(n_coupons, n_trials=5000):
    """Simulate coupon collector problem."""
    results = []
    for _ in range(n_trials):
        collected = set()
        boxes = 0
        while len(collected) < n_coupons:
            collected.add(np.random.randint(0, n_coupons))
            boxes += 1
        results.append(boxes)
    return results

n_coupons_list = [5, 10, 20, 50]
expected_theory = [n * sum(1/k for k in range(1, n+1)) for n in n_coupons_list]

sim_means = []
for n in n_coupons_list:
    results = coupon_collector_sim(n, n_trials=3000)
    sim_means.append(np.mean(results))

x_pos = range(len(n_coupons_list))
width = 0.35
bars1 = axes[2].bar([p - width/2 for p in x_pos], expected_theory, width, 
                     label='Theory: n·H(n)', color='#3498db', edgecolor='black')
bars2 = axes[2].bar([p + width/2 for p in x_pos], sim_means, width,
                     label='Simulation', color='#e67e22', edgecolor='black')
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels([f'n={n}' for n in n_coupons_list])
axes[2].set_ylabel('Expected # of Boxes')
axes[2].set_title('Coupon Collector Problem', fontsize=12)
axes[2].legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"\nBirthday Problem: P(match | 23 people) = {birthday_probability(23):.6f}")
print(f"Coupon Collector (n=50): Theory = {50*sum(1/k for k in range(1,51)):.1f}, Sim = {sim_means[3]:.1f}")

# 2. Random Variables & Distributions

---

## 2.1 Random Variables

A **random variable** $$X$$ is a measurable function from the sample space $$\Omega$$ to $$\mathbb{R}$$:
$$X: \Omega \to \mathbb{R}$$

### Discrete Random Variables
Takes countable values. Characterized by the **Probability Mass Function (PMF)**:
$$p_X(x) = P(X = x), \quad \sum_x p_X(x) = 1$$

### Continuous Random Variables  
Takes uncountable values. Characterized by the **Probability Density Function (PDF)**:
$$f_X(x) \geq 0, \quad \int_{-\infty}^{\infty} f_X(x) \, dx = 1$$

$$P(a \leq X \leq b) = \int_a^b f_X(x) \, dx$$

> ⚠️ **Key**: For continuous RVs, $$P(X = x) = 0$$ for any specific value $$x$$.

### Cumulative Distribution Function (CDF)
$$F_X(x) = P(X \leq x)$$

Properties:
* Non-decreasing, right-continuous
* $$\lim_{x \to -\infty} F(x) = 0$$, $$\lim_{x \to \infty} F(x) = 1$$
* $$P(a < X \leq b) = F(b) - F(a)$$

---

## 2.2 Key Discrete Distributions

### Bernoulli($$p$$)
$$X \in \{0, 1\}$$, $$P(X=1) = p$$

$$E[X] = p, \quad \text{Var}(X) = p(1-p)$$

**Example**: A single ad click (click or no click).

---

### Binomial($$n, p$$)
Number of successes in $$n$$ independent Bernoulli trials:

$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}, \quad k = 0, 1, \ldots, n$$

$$E[X] = np, \quad \text{Var}(X) = np(1-p)$$

**Example**: Out of 1000 users shown an ad, how many click? (Conversion rate = $$p$$)

---

### Poisson($$\lambda$$)
Counts rare events in a fixed interval. Arises as limit of Binomial when $$n \to \infty$$, $$p \to 0$$, $$np \to \lambda$$:

$$P(X = k) = \frac{\lambda^k e^{-\lambda}}{k!}, \quad k = 0, 1, 2, \ldots$$

$$E[X] = \lambda, \quad \text{Var}(X) = \lambda$$

**Key Property**: $$\text{Mean} = \text{Variance}$$ (used to test Poisson assumption)

**Example**: Number of server crashes per day, customer support tickets per hour, fraud transactions per week.

---

### Geometric($$p$$)
Number of trials until first success:

$$P(X = k) = (1-p)^{k-1} p, \quad k = 1, 2, 3, \ldots$$

$$E[X] = 1/p, \quad \text{Var}(X) = (1-p)/p^2$$

**Memoryless property**: $$P(X > s + t | X > s) = P(X > t)$$

**Example**: Number of job applications before getting an offer.

---

### Negative Binomial($$r, p$$)
Number of trials until $$r$$-th success:

$$P(X = k) = \binom{k-1}{r-1} p^r (1-p)^{k-r}, \quad k = r, r+1, \ldots$$

$$E[X] = r/p, \quad \text{Var}(X) = r(1-p)/p^2$$

**Example**: Modeling overdispersed count data (when Poisson's mean=variance is too restrictive).

---

## 2.3 Key Continuous Distributions

### Uniform($$a, b$$)
$$f(x) = \frac{1}{b-a}, \quad a \leq x \leq b$$

$$E[X] = \frac{a+b}{2}, \quad \text{Var}(X) = \frac{(b-a)^2}{12}$$

---

### Normal (Gaussian) $$N(\mu, \sigma^2)$$

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

$$E[X] = \mu, \quad \text{Var}(X) = \sigma^2$$

**Properties:**
* Symmetric about $$\mu$$
* **68-95-99.7 Rule**: ~68% within $$\pm 1\sigma$$, ~95% within $$\pm 2\sigma$$, ~99.7% within $$\pm 3\sigma$$
* Sum of normals is normal: $$X \sim N(\mu_1, \sigma_1^2), Y \sim N(\mu_2, \sigma_2^2) \Rightarrow X + Y \sim N(\mu_1 + \mu_2, \sigma_1^2 + \sigma_2^2)$$
* **MGF**: $$M_X(t) = \exp(\mu t + \sigma^2 t^2 / 2)$$

**Why it appears everywhere**: Central Limit Theorem — the sum/average of many independent random variables tends toward Normal.

---

### Exponential($$\lambda$$)
Time between events in a Poisson process:

$$f(x) = \lambda e^{-\lambda x}, \quad x \geq 0$$

$$E[X] = 1/\lambda, \quad \text{Var}(X) = 1/\lambda^2$$

**Memoryless property**: $$P(X > s + t | X > s) = P(X > t)$$

**Example**: Time between customer arrivals, time to next server failure.

---

### Gamma($$\alpha, \beta$$)
Generalizes exponential. Sum of $$\alpha$$ exponentials:

$$f(x) = \frac{\beta^\alpha}{\Gamma(\alpha)} x^{\alpha-1} e^{-\beta x}, \quad x > 0$$

$$E[X] = \alpha/\beta, \quad \text{Var}(X) = \alpha/\beta^2$$

**Special cases**: $$\text{Gamma}(1, \lambda) = \text{Exp}(\lambda)$$, $$\text{Gamma}(n/2, 1/2) = \chi^2(n)$$

---

### Beta($$\alpha, \beta$$)
Distribution on $$[0, 1]$$ — perfect for modeling probabilities:

$$f(x) = \frac{x^{\alpha-1}(1-x)^{\beta-1}}{B(\alpha, \beta)}, \quad 0 \leq x \leq 1$$

where $$B(\alpha, \beta) = \frac{\Gamma(\alpha)\Gamma(\beta)}{\Gamma(\alpha+\beta)}$$

$$E[X] = \frac{\alpha}{\alpha+\beta}, \quad \text{Var}(X) = \frac{\alpha\beta}{(\alpha+\beta)^2(\alpha+\beta+1)}$$

**Conjugate prior for Binomial** — fundamental in Bayesian A/B testing.

In [0]:
from scipy import stats

# =============================================================================
# COMPREHENSIVE DISTRIBUTION VISUALIZATIONS
# =============================================================================

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# --- Binomial ---
ax = axes[0, 0]
for n, p in [(20, 0.3), (20, 0.5), (50, 0.7)]:
    x = np.arange(0, n+1)
    pmf = stats.binom.pmf(x, n, p)
    ax.plot(x, pmf, 'o-', markersize=3, label=f'n={n}, p={p}')
ax.set_title('Binomial Distribution', fontsize=12, fontweight='bold')
ax.set_xlabel('k')
ax.set_ylabel('P(X=k)')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Poisson ---
ax = axes[0, 1]
for lam in [1, 4, 10, 20]:
    x = np.arange(0, 35)
    pmf = stats.poisson.pmf(x, lam)
    ax.plot(x, pmf, 'o-', markersize=3, label=f'λ={lam}')
ax.set_title('Poisson Distribution', fontsize=12, fontweight='bold')
ax.set_xlabel('k')
ax.set_ylabel('P(X=k)')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Normal ---
ax = axes[0, 2]
x = np.linspace(-6, 6, 1000)
for mu, sigma in [(0, 1), (0, 2), (2, 1), (-2, 0.5)]:
    pdf = stats.norm.pdf(x, mu, sigma)
    ax.plot(x, pdf, linewidth=2, label=f'μ={mu}, σ={sigma}')
ax.set_title('Normal Distribution', fontsize=12, fontweight='bold')
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.legend()
ax.grid(True, alpha=0.3)

# Shade 68-95-99.7 rule for standard normal
ax_inset = ax
x_fill = np.linspace(-3, 3, 1000)
ax.fill_between(x_fill, stats.norm.pdf(x_fill, 0, 1), alpha=0.05, color='blue')

# --- Exponential ---
ax = axes[1, 0]
x = np.linspace(0, 5, 1000)
for lam in [0.5, 1, 2, 5]:
    pdf = stats.expon.pdf(x, scale=1/lam)
    ax.plot(x, pdf, linewidth=2, label=f'λ={lam}')
ax.set_title('Exponential Distribution', fontsize=12, fontweight='bold')
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Beta ---
ax = axes[1, 1]
x = np.linspace(0.001, 0.999, 1000)
beta_params = [(0.5, 0.5), (1, 1), (2, 5), (5, 2), (5, 5), (10, 2)]
for a, b in beta_params:
    pdf = stats.beta.pdf(x, a, b)
    ax.plot(x, pdf, linewidth=2, label=f'α={a}, β={b}')
ax.set_title('Beta Distribution', fontsize=12, fontweight='bold')
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.legend(fontsize=9)
ax.set_ylim(0, 5)
ax.grid(True, alpha=0.3)

# --- Gamma ---
ax = axes[1, 2]
x = np.linspace(0.01, 20, 1000)
for a, b in [(1, 1), (2, 1), (3, 1), (5, 1), (9, 0.5)]:
    pdf = stats.gamma.pdf(x, a, scale=1/b)
    ax.plot(x, pdf, linewidth=2, label=f'α={a}, β={b}')
ax.set_title('Gamma Distribution', fontsize=12, fontweight='bold')
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Key Probability Distributions for Data Science Interviews', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 2.4 Expectation, Variance & Moments

### Expectation (Mean)

$$E[X] = \begin{cases} \sum_x x \cdot p(x) & \text{discrete} \\ \int_{-\infty}^{\infty} x \cdot f(x) \, dx & \text{continuous} \end{cases}$$

**Properties (Linearity):**
* $$E[aX + b] = aE[X] + b$$
* $$E[X + Y] = E[X] + E[Y]$$ (ALWAYS, even if dependent!)
* $$E[XY] = E[X]E[Y]$$ only if $$X, Y$$ are independent

**Law of the Unconscious Statistician (LOTUS):**
$$E[g(X)] = \int g(x) f_X(x) \, dx$$

---

### Variance

$$\text{Var}(X) = E[(X - \mu)^2] = E[X^2] - (E[X])^2$$

**Properties:**
* $$\text{Var}(aX + b) = a^2 \text{Var}(X)$$
* $$\text{Var}(X + Y) = \text{Var}(X) + \text{Var}(Y) + 2\text{Cov}(X, Y)$$
* If independent: $$\text{Var}(X + Y) = \text{Var}(X) + \text{Var}(Y)$$

---

### Covariance & Correlation

$$\text{Cov}(X, Y) = E[(X - \mu_X)(Y - \mu_Y)] = E[XY] - E[X]E[Y]$$

$$\rho_{XY} = \frac{\text{Cov}(X, Y)}{\sigma_X \sigma_Y}, \quad -1 \leq \rho \leq 1$$

> ⚠️ **Interview Trap**: $$\rho = 0$$ means uncorrelated, NOT independent! (Exception: jointly normal RVs)

---

### Moment Generating Functions (MGF)

$$M_X(t) = E[e^{tX}]$$

**Why MGFs matter:**
1. The $$n$$-th moment: $$E[X^n] = M_X^{(n)}(0) = \frac{d^n}{dt^n} M_X(t) \Big|_{t=0}$$
2. **Uniqueness**: If two RVs have the same MGF, they have the same distribution
3. **Sum of independents**: $$M_{X+Y}(t) = M_X(t) \cdot M_Y(t)$$

**Proof that sum of Normals is Normal:**

$$M_{X+Y}(t) = M_X(t) \cdot M_Y(t) = e^{\mu_1 t + \sigma_1^2 t^2/2} \cdot e^{\mu_2 t + \sigma_2^2 t^2/2} = e^{(\mu_1+\mu_2)t + (\sigma_1^2+\sigma_2^2)t^2/2}$$

This is the MGF of $$N(\mu_1 + \mu_2, \sigma_1^2 + \sigma_2^2)$$. ■

---

### Skewness & Kurtosis

**Skewness** (3rd standardized moment): $$\gamma_1 = E\left[\left(\frac{X-\mu}{\sigma}\right)^3\right]$$

* $$\gamma_1 > 0$$: right-skewed (income, house prices)
* $$\gamma_1 < 0$$: left-skewed (exam scores with ceiling effect)

**Kurtosis** (4th standardized moment): $$\kappa = E\left[\left(\frac{X-\mu}{\sigma}\right)^4\right]$$

* Normal has $$\kappa = 3$$ (excess kurtosis = 0)
* $$\kappa > 3$$: heavy tails (financial returns)
* $$\kappa < 3$$: light tails

# 3. Central Limit Theorem & Law of Large Numbers

---

## 3.1 Law of Large Numbers (LLN)

Let $$X_1, X_2, \ldots$$ be i.i.d. with $$E[X_i] = \mu$$ and $$\text{Var}(X_i) = \sigma^2 < \infty$$.

### Weak Law (WLLN)
For any $$\epsilon > 0$$:
$$P\left(\left|\bar{X}_n - \mu\right| > \epsilon\right) \to 0 \quad \text{as } n \to \infty$$

(Convergence in probability)

### Strong Law (SLLN)
$$P\left(\lim_{n \to \infty} \bar{X}_n = \mu\right) = 1$$

(Almost sure convergence)

**Product Intuition**: As you collect more user data, your sample metrics converge to the true population values. This is why larger experiments give more reliable results.

---

## 3.2 Central Limit Theorem (CLT)

**The most important theorem in statistics.**

Let $$X_1, X_2, \ldots, X_n$$ be i.i.d. with $$E[X_i] = \mu$$, $$\text{Var}(X_i) = \sigma^2 < \infty$$. Then:

$$\frac{\bar{X}_n - \mu}{\sigma / \sqrt{n}} \xrightarrow{d} N(0, 1) \quad \text{as } n \to \infty$$

Equivalently:
$$\bar{X}_n \approx N\left(\mu, \frac{\sigma^2}{n}\right) \quad \text{for large } n$$

Or for the sum:
$$S_n = \sum_{i=1}^n X_i \approx N(n\mu, n\sigma^2)$$

### Why CLT Matters in Data Science:
1. **Justifies z-tests and t-tests** for large samples regardless of underlying distribution
2. **Confidence intervals** rely on approximate normality of $$\bar{X}$$
3. **A/B testing** uses CLT to compare sample means
4. **Error bounds** — precision scales as $$1/\sqrt{n}$$ (diminishing returns!)

### When does CLT NOT apply?
* Infinite variance distributions (Cauchy, Pareto with $$\alpha \leq 2$$)
* Strong dependence between observations
* Extreme values / heavy tails (need $$n$$ very large)

---

## 3.3 Berry-Esseen Theorem (Rate of Convergence)

The CLT tells us convergence happens, but how fast?

$$\sup_x \left| P\left(\frac{\bar{X}_n - \mu}{\sigma/\sqrt{n}} \leq x\right) - \Phi(x)\right| \leq \frac{C \cdot \rho}{\sigma^3 \sqrt{n}}$$

where $$\rho = E[|X - \mu|^3]$$ and $$C \leq 0.4748$$.

**Implication**: The more skewed the distribution (larger $$\rho/\sigma^3$$), the larger $$n$$ needs to be for CLT to kick in.

---

## 3.4 Delta Method

If $$\sqrt{n}(\bar{X}_n - \mu) \xrightarrow{d} N(0, \sigma^2)$$ and $$g$$ is differentiable at $$\mu$$ with $$g'(\mu) \neq 0$$:

$$\sqrt{n}(g(\bar{X}_n) - g(\mu)) \xrightarrow{d} N(0, \sigma^2 [g'(\mu)]^2)$$

**Application**: Finding the variance of $$\log(\bar{X})$$, ratios of means, etc.

**Example**: If $$\bar{X} \sim N(\mu, \sigma^2/n)$$ and we want the distribution of $$g(\bar{X}) = 1/\bar{X}$$:

$$\text{Var}(1/\bar{X}) \approx \frac{\sigma^2}{n \mu^4}$$

In [0]:
# =============================================================================
# CENTRAL LIMIT THEOREM: Visual Proof
# Showing CLT works regardless of the parent distribution
# =============================================================================

np.random.seed(42)

# Different parent distributions (all with finite variance)
distributions = {
    'Exponential(λ=1)\n(heavily right-skewed)': lambda n: np.random.exponential(1, n),
    'Uniform(0,1)\n(flat, no skew)': lambda n: np.random.uniform(0, 1, n),
    'Bernoulli(p=0.3)\n(discrete, skewed)': lambda n: np.random.binomial(1, 0.3, n),
    'Chi-squared(df=3)\n(right-skewed)': lambda n: np.random.chisquare(3, n),
}

sample_sizes = [1, 2, 5, 30, 100]
n_simulations = 10000

fig, axes = plt.subplots(len(distributions), len(sample_sizes), figsize=(18, 12))

for row, (dist_name, sampler) in enumerate(distributions.items()):
    for col, n in enumerate(sample_sizes):
        # Generate n_simulations sample means, each from a sample of size n
        sample_means = [np.mean(sampler(n)) for _ in range(n_simulations)]
        
        ax = axes[row, col]
        ax.hist(sample_means, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='white')
        
        # Overlay theoretical normal (from CLT)
        if n > 1:
            x_range = np.linspace(min(sample_means), max(sample_means), 200)
            mu_hat = np.mean(sample_means)
            sigma_hat = np.std(sample_means)
            ax.plot(x_range, stats.norm.pdf(x_range, mu_hat, sigma_hat), 
                   'r-', linewidth=2, label='Normal fit')
        
        if row == 0:
            ax.set_title(f'n = {n}', fontsize=12, fontweight='bold')
        if col == 0:
            ax.set_ylabel(dist_name, fontsize=9, fontweight='bold')
        
        ax.set_yticks([])
        if col == len(sample_sizes) - 1 and row == 0:
            ax.legend(fontsize=8)

plt.suptitle('Central Limit Theorem: Distribution of Sample Mean $\\bar{X}_n$\n'
             'As n increases, the distribution of $\\bar{X}_n$ becomes Normal regardless of parent distribution',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("KEY TAKEAWAY: Even for highly non-normal distributions (exponential,")
print("Bernoulli, chi-squared), the sample mean becomes approximately normal")
print("by n=30. This is why n=30 is the classic 'rule of thumb' for CLT.")
print("="*70)

# 4. Estimation Theory

---

## 4.1 Properties of Estimators

An **estimator** $$\hat{\theta}$$ is a function of the data used to estimate an unknown parameter $$\theta$$.

### Bias
$$\text{Bias}(\hat{\theta}) = E[\hat{\theta}] - \theta$$

An estimator is **unbiased** if $$E[\hat{\theta}] = \theta$$.

**Example**: $$\bar{X}$$ is unbiased for $$\mu$$, but $$\frac{1}{n}\sum(X_i - \bar{X})^2$$ is biased for $$\sigma^2$$ (Bessel's correction: divide by $$n-1$$).

### Mean Squared Error
$$\text{MSE}(\hat{\theta}) = E[(\hat{\theta} - \theta)^2] = \text{Var}(\hat{\theta}) + [\text{Bias}(\hat{\theta})]^2$$

> ⚠️ **Key Insight**: A biased estimator can have LOWER MSE than an unbiased one if it has much lower variance (bias-variance tradeoff for estimators!).

### Consistency
$$\hat{\theta}_n \xrightarrow{P} \theta$$ as $$n \to \infty$$

Sufficient condition: $$\text{MSE}(\hat{\theta}_n) \to 0$$

### Efficiency & Cramér-Rao Lower Bound

The variance of ANY unbiased estimator is bounded below:

$$\text{Var}(\hat{\theta}) \geq \frac{1}{n \cdot I(\theta)}$$

where $$I(\theta)$$ is the **Fisher Information**:

$$I(\theta) = -E\left[\frac{\partial^2 \ln f(X|\theta)}{\partial \theta^2}\right] = E\left[\left(\frac{\partial \ln f(X|\theta)}{\partial \theta}\right)^2\right]$$

An estimator achieving this bound is called **efficient** (MLE is asymptotically efficient).

---

## 4.2 Maximum Likelihood Estimation (MLE)

Given data $$x_1, \ldots, x_n$$ from $$f(x|\theta)$$, the **likelihood function** is:

$$L(\theta) = \prod_{i=1}^n f(x_i | \theta)$$

The **log-likelihood** (easier to work with):

$$\ell(\theta) = \sum_{i=1}^n \ln f(x_i | \theta)$$

The **MLE** maximizes the likelihood:

$$\hat{\theta}_{MLE} = \arg\max_\theta \ell(\theta)$$

Found by solving the **score equation**: $$\frac{\partial \ell}{\partial \theta} = 0$$

### Properties of MLE (asymptotic):
1. **Consistent**: $$\hat{\theta}_{MLE} \xrightarrow{P} \theta_0$$
2. **Asymptotically Normal**: $$\sqrt{n}(\hat{\theta}_{MLE} - \theta_0) \xrightarrow{d} N(0, I(\theta_0)^{-1})$$
3. **Asymptotically Efficient**: Achieves Cramér-Rao bound
4. **Invariance**: If $$\hat{\theta}$$ is MLE of $$\theta$$, then $$g(\hat{\theta})$$ is MLE of $$g(\theta)$$

### Example 1: MLE for Normal Distribution

Given $$X_i \sim N(\mu, \sigma^2)$$:

$$\ell(\mu, \sigma^2) = -\frac{n}{2}\ln(2\pi) - \frac{n}{2}\ln(\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^n(x_i - \mu)^2$$

Taking derivatives:
* $$\hat{\mu}_{MLE} = \bar{X}$$ (sample mean)
* $$\hat{\sigma}^2_{MLE} = \frac{1}{n}\sum(X_i - \bar{X})^2$$ (biased! but consistent)

### Example 2: MLE for Bernoulli

$$\ell(p) = \sum x_i \ln p + \sum(1-x_i)\ln(1-p)$$

$$\frac{\partial \ell}{\partial p} = \frac{\sum x_i}{p} - \frac{n - \sum x_i}{1-p} = 0$$

$$\hat{p}_{MLE} = \bar{X} = \frac{\text{number of successes}}{n}$$

---

## 4.3 Method of Moments (MoM)

Set sample moments equal to population moments and solve:

$$\frac{1}{n}\sum_{i=1}^n X_i^k = E[X^k] \quad \text{for } k = 1, 2, \ldots$$

**Advantage over MLE**: Simpler computation, doesn't need full distributional form  
**Disadvantage**: Less efficient (higher variance) than MLE

In [0]:
from scipy.optimize import minimize_scalar, minimize

# =============================================================================
# MAXIMUM LIKELIHOOD ESTIMATION: From Scratch
# =============================================================================

np.random.seed(42)

# --- Example 1: MLE for Exponential Distribution ---
# True parameter
lambda_true = 2.5
n_samples = 500
data_exp = np.random.exponential(1/lambda_true, n_samples)

# Analytical MLE: lambda_hat = 1 / x_bar
lambda_mle_analytical = 1 / np.mean(data_exp)

# Numerical MLE (for verification)
def neg_log_likelihood_exp(lam, data):
    if lam <= 0:
        return np.inf
    return -np.sum(np.log(lam) - lam * data)

result = minimize_scalar(neg_log_likelihood_exp, bounds=(0.01, 10), 
                         method='bounded', args=(data_exp,))
lambda_mle_numerical = result.x

print("=" * 60)
print("MLE FOR EXPONENTIAL DISTRIBUTION")
print("=" * 60)
print(f"True λ:           {lambda_true}")
print(f"Analytical MLE:    {lambda_mle_analytical:.4f}")
print(f"Numerical MLE:     {lambda_mle_numerical:.4f}")
print(f"Sample size:       {n_samples}")

# --- Example 2: MLE for Normal Distribution ---
mu_true, sigma_true = 5.0, 2.0
data_norm = np.random.normal(mu_true, sigma_true, n_samples)

mu_mle = np.mean(data_norm)
sigma_mle = np.std(data_norm)  # MLE (biased)
sigma_unbiased = np.std(data_norm, ddof=1)  # Bessel's correction

print(f"\n{'='*60}")
print("MLE FOR NORMAL DISTRIBUTION")
print("="*60)
print(f"True μ={mu_true}, σ={sigma_true}")
print(f"MLE:     μ̂={mu_mle:.4f}, σ̂={sigma_mle:.4f}")
print(f"Unbiased: μ̂={mu_mle:.4f}, s={sigma_unbiased:.4f}")

# --- Visualize Likelihood Surface for Exponential ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Log-likelihood for exponential
lambdas = np.linspace(0.5, 5.0, 1000)
log_liks = [-neg_log_likelihood_exp(l, data_exp) for l in lambdas]

axes[0].plot(lambdas, log_liks, 'b-', linewidth=2)
axes[0].axvline(x=lambda_mle_analytical, color='r', linestyle='--', 
                label=f'MLE = {lambda_mle_analytical:.3f}', linewidth=2)
axes[0].axvline(x=lambda_true, color='g', linestyle='--', 
                label=f'True λ = {lambda_true}', linewidth=2)
axes[0].set_xlabel('λ', fontsize=12)
axes[0].set_ylabel('Log-Likelihood ℓ(λ)', fontsize=12)
axes[0].set_title('MLE for Exponential: Log-Likelihood', fontsize=12)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Plot 2: MLE convergence with sample size
sample_sizes = np.arange(5, 1001, 5)
mle_estimates = [1/np.mean(data_exp[:n]) for n in sample_sizes]

axes[1].plot(sample_sizes, mle_estimates, 'b-', alpha=0.7, linewidth=1)
axes[1].axhline(y=lambda_true, color='r', linestyle='--', linewidth=2, label=f'True λ={lambda_true}')
axes[1].fill_between(sample_sizes, 
                     lambda_true - 1.96*lambda_true/np.sqrt(sample_sizes),
                     lambda_true + 1.96*lambda_true/np.sqrt(sample_sizes),
                     alpha=0.2, color='red', label='95% CI (asymptotic)')
axes[1].set_xlabel('Sample Size n', fontsize=12)
axes[1].set_ylabel('MLE Estimate λ̂', fontsize=12)
axes[1].set_title('MLE Consistency: Convergence to True Value', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# Plot 3: Fisher Information and efficiency
# For Exponential: I(λ) = 1/λ², so Var(MLE) ≥ λ²/n
n_reps = 5000
sample_sizes_fi = [10, 30, 50, 100, 200, 500]
empirical_vars = []
cramer_rao_bounds = []

for n in sample_sizes_fi:
    mle_reps = [1/np.mean(np.random.exponential(1/lambda_true, n)) for _ in range(n_reps)]
    empirical_vars.append(np.var(mle_reps))
    cramer_rao_bounds.append(lambda_true**2 / n)  # CRLB for exponential

axes[2].plot(sample_sizes_fi, empirical_vars, 'bo-', label='Empirical Var(MLE)', linewidth=2)
axes[2].plot(sample_sizes_fi, cramer_rao_bounds, 'r--', label='Cramér-Rao Lower Bound', linewidth=2)
axes[2].set_xlabel('Sample Size n', fontsize=12)
axes[2].set_ylabel('Variance', fontsize=12)
axes[2].set_title('MLE Efficiency: Var vs Cramér-Rao Bound', fontsize=12)
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n\n→ KEY INSIGHT: MLE achieves the Cramér-Rao lower bound asymptotically,")
print("  making it the most efficient estimator for large samples.")

# 5. Hypothesis Testing

---

## 5.1 Framework

**Null Hypothesis** $$H_0$$: Status quo (no effect, no difference)  
**Alternative Hypothesis** $$H_1$$ (or $$H_a$$): The claim we're testing

### Decision Framework:

| | $$H_0$$ True | $$H_0$$ False |
|---|---|---|
| **Reject $$H_0$$** | Type I Error ($$\alpha$$) | Correct! (Power = $$1 - \beta$$) |
| **Fail to Reject** | Correct! | Type II Error ($$\beta$$) |

* **Significance level** $$\alpha$$: $$P(\text{Reject } H_0 | H_0 \text{ true})$$ — false positive rate
* **Power** $$1 - \beta$$: $$P(\text{Reject } H_0 | H_0 \text{ false})$$ — ability to detect a real effect
* **p-value**: $$P(\text{data as extreme or more} | H_0 \text{ true})$$

> ⚠️ **Critical Interview Distinction**: A p-value is NOT "the probability that $$H_0$$ is true." It's the probability of the observed data (or more extreme) UNDER the assumption that $$H_0$$ IS true.

---

## 5.2 Z-Test (Known Variance)

Test $$H_0: \mu = \mu_0$$ vs $$H_1: \mu \neq \mu_0$$ when $$\sigma$$ is known:

$$Z = \frac{\bar{X} - \mu_0}{\sigma / \sqrt{n}} \sim N(0,1) \text{ under } H_0$$

Reject $$H_0$$ if $$|Z| > z_{\alpha/2}$$ (two-sided)

---

## 5.3 T-Test (Unknown Variance)

When $$\sigma$$ is unknown (the realistic case), replace with sample $$s$$:

$$T = \frac{\bar{X} - \mu_0}{s / \sqrt{n}} \sim t_{n-1} \text{ under } H_0$$

The $$t$$-distribution has heavier tails than Normal (accounts for uncertainty in $$s$$). As $$n \to \infty$$, $$t_n \to N(0,1)$$.

### Two-Sample T-Test (Equal Variances)
$$T = \frac{\bar{X}_1 - \bar{X}_2}{s_p \sqrt{1/n_1 + 1/n_2}}$$

where $$s_p = \sqrt{\frac{(n_1-1)s_1^2 + (n_2-1)s_2^2}{n_1 + n_2 - 2}}$$ (pooled std)

### Welch's T-Test (Unequal Variances) — ★ Preferred in practice
$$T = \frac{\bar{X}_1 - \bar{X}_2}{\sqrt{s_1^2/n_1 + s_2^2/n_2}}$$

df approximated by Satterthwaite formula.

---

## 5.4 Chi-Squared Tests

### Goodness of Fit
Test whether observed frequencies match expected:
$$\chi^2 = \sum_{i=1}^k \frac{(O_i - E_i)^2}{E_i} \sim \chi^2_{k-1}$$

### Test of Independence (Contingency Table)
For an $$r \times c$$ table:
$$\chi^2 = \sum_{i=1}^r \sum_{j=1}^c \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$

where $$E_{ij} = \frac{(\text{row } i \text{ total})(\text{col } j \text{ total})}{n}$$

df = $$(r-1)(c-1)$$

**Product Example**: Testing whether user conversion rate is independent of the device type (mobile/desktop/tablet).

---

## 5.5 Multiple Testing Problem

When conducting $$m$$ tests at level $$\alpha$$:
$$P(\text{at least one false positive}) = 1 - (1-\alpha)^m$$

For $$m = 20$$ tests at $$\alpha = 0.05$$: $$P \approx 0.64$$ — almost certain to get a spurious result!

### Bonferroni Correction
Use $$\alpha/m$$ for each test. Conservative but simple.

### Benjamini-Hochberg (FDR Control)
Controls the **False Discovery Rate** = $$E\left[\frac{\text{false positives}}{\text{total rejections}}\right]$$

Algorithm:
1. Sort p-values: $$p_{(1)} \leq p_{(2)} \leq \cdots \leq p_{(m)}$$
2. Find largest $$k$$ such that $$p_{(k)} \leq \frac{k}{m} \cdot \alpha$$
3. Reject all $$H_{(1)}, \ldots, H_{(k)}$$

---

## 5.6 Power Analysis

Statistical power depends on:
1. **Effect size** ($$\delta$$): Larger effect → easier to detect
2. **Sample size** ($$n$$): More data → more power
3. **Significance level** ($$\alpha$$): Larger $$\alpha$$ → more power (but more false positives)
4. **Variance** ($$\sigma^2$$): Less noise → more power

For a two-sample z-test with equal groups:

$$n = \frac{2(z_{\alpha/2} + z_\beta)^2 \sigma^2}{\delta^2}$$

where $$\delta = \mu_1 - \mu_2$$ is the minimum detectable effect.

In [0]:
from scipy.stats import ttest_ind, ttest_1samp, chi2_contingency, norm, t
from statsmodels.stats.power import TTestIndPower
from statsmodels.stats.multitest import multipletests

# =============================================================================
# HYPOTHESIS TESTING: Complete Workflow
# =============================================================================

np.random.seed(42)

# --- SCENARIO: A/B Test at an E-commerce Company ---
# Control group: existing checkout flow
# Treatment group: new streamlined checkout
# Metric: Revenue per user (in dollars)

n_control = 5000
n_treatment = 5000

# Simulating data with a real effect (treatment increases revenue by $2)
control = np.random.lognormal(mean=3.0, sigma=1.2, size=n_control)
treatment = np.random.lognormal(mean=3.0, sigma=1.2, size=n_treatment) * 1.05  # 5% lift

print("=" * 70)
print("HYPOTHESIS TEST: Does the new checkout flow increase revenue per user?")
print("=" * 70)
print(f"\nControl:   n={n_control}, mean=${np.mean(control):.2f}, std=${np.std(control):.2f}")
print(f"Treatment: n={n_treatment}, mean=${np.mean(treatment):.2f}, std=${np.std(treatment):.2f}")
print(f"Observed lift: ${np.mean(treatment) - np.mean(control):.2f} ({(np.mean(treatment)/np.mean(control)-1)*100:.2f}%)")

# Welch's t-test (unequal variances assumed)
t_stat, p_value = ttest_ind(treatment, control, equal_var=False)

print(f"\n--- Welch's Two-Sample T-Test ---")
print(f"H₀: μ_treatment = μ_control (no difference)")
print(f"H₁: μ_treatment ≠ μ_control (two-sided)")
print(f"\nTest statistic: t = {t_stat:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"\nDecision at α=0.05: {'REJECT H₀ ✔' if p_value < 0.05 else 'FAIL TO REJECT H₀'}")
print(f"Conclusion: {'Statistically significant difference detected!' if p_value < 0.05 else 'No significant difference.'}")

# Confidence interval for the difference
diff = np.mean(treatment) - np.mean(control)
se_diff = np.sqrt(np.var(treatment)/n_treatment + np.var(control)/n_control)
ci_lower = diff - 1.96 * se_diff
ci_upper = diff + 1.96 * se_diff
print(f"\n95% CI for difference: [${ci_lower:.2f}, ${ci_upper:.2f}]")

# --- VISUALIZATION ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Distribution comparison
axes[0].hist(control, bins=50, alpha=0.6, density=True, label=f'Control (n={n_control})', color='steelblue')
axes[0].hist(treatment, bins=50, alpha=0.6, density=True, label=f'Treatment (n={n_treatment})', color='coral')
axes[0].axvline(np.mean(control), color='blue', linestyle='--', linewidth=2)
axes[0].axvline(np.mean(treatment), color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Revenue per User ($)', fontsize=11)
axes[0].set_ylabel('Density', fontsize=11)
axes[0].set_title('Revenue Distributions', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].set_xlim(0, 200)

# Plot 2: Power curve
power_analysis = TTestIndPower()
effect_sizes = np.linspace(0.01, 0.5, 100)
powers = [power_analysis.power(effect_size=es, nobs1=5000, ratio=1.0, alpha=0.05) 
          for es in effect_sizes]

axes[1].plot(effect_sizes, powers, 'b-', linewidth=2)
axes[1].axhline(y=0.8, color='r', linestyle='--', alpha=0.7, label='80% power threshold')
axes[1].fill_between(effect_sizes, powers, alpha=0.1)
axes[1].set_xlabel("Cohen's d (Effect Size)", fontsize=11)
axes[1].set_ylabel('Statistical Power', fontsize=11)
axes[1].set_title('Power Curve (n=5000 per group, α=0.05)', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# Find minimum detectable effect at 80% power
mde = power_analysis.solve_power(power=0.8, nobs1=5000, ratio=1.0, alpha=0.05)
axes[1].axvline(x=mde, color='g', linestyle='--', alpha=0.7)
axes[1].annotate(f'MDE = {mde:.3f}', xy=(mde, 0.8), xytext=(mde+0.05, 0.6),
                fontsize=10, arrowprops=dict(arrowstyle='->'))

# Plot 3: p-value distribution under H0 (should be uniform!)
n_simulations = 10000
p_values_null = []
for _ in range(n_simulations):
    # Both drawn from same distribution (H0 is true)
    x = np.random.normal(50, 10, 100)
    y = np.random.normal(50, 10, 100)
    _, p = ttest_ind(x, y)
    p_values_null.append(p)

axes[2].hist(p_values_null, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='white')
axes[2].axhline(y=1.0, color='r', linestyle='--', linewidth=2, label='Uniform(0,1)')
axes[2].axvline(x=0.05, color='orange', linestyle='--', linewidth=2, label='α=0.05')
axes[2].set_xlabel('p-value', fontsize=11)
axes[2].set_ylabel('Density', fontsize=11)
axes[2].set_title(f'P-values Under H₀ (should be Uniform)\nFalse positive rate: {np.mean(np.array(p_values_null)<0.05):.3f}', fontsize=11)
axes[2].legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"\n→ Under H₀, exactly α={np.mean(np.array(p_values_null)<0.05):.1%} of p-values fall below 0.05")
print(f"  (This IS the definition of Type I error rate!)")

# 6. A/B Testing & Experimental Design

---

## 6.1 A/B Testing Framework

A/B testing is the gold standard for causal inference in tech companies. It's a randomized controlled experiment.

### Key Components:
1. **Randomization unit**: User, session, page view, device
2. **Primary metric**: The KPI you're optimizing (conversion rate, revenue, engagement)
3. **Guardrail metrics**: Metrics that should NOT degrade (latency, crash rate)
4. **Sample size**: Determined by power analysis BEFORE the experiment
5. **Duration**: Long enough to capture weekly cycles (typically 1-2 weeks minimum)

---

## 6.2 Sample Size Calculation

### For Proportions (Conversion Rates)

To detect a difference $$\delta = p_1 - p_2$$ with power $$1 - \beta$$:

$$n = \frac{(z_{\alpha/2} + z_\beta)^2 \cdot [p_1(1-p_1) + p_2(1-p_2)]}{(p_1 - p_2)^2}$$

Simplified (assuming $$p_1 \approx p_2 \approx p$$):

$$n \approx \frac{2p(1-p)(z_{\alpha/2} + z_\beta)^2}{\delta^2}$$

### For Continuous Metrics (Revenue, Time Spent)

$$n = \frac{2\sigma^2(z_{\alpha/2} + z_\beta)^2}{\delta^2}$$

---

## 6.3 Variance Reduction Techniques

Reducing variance = needing fewer users = faster experiments!

### CUPED (Controlled-experiment Using Pre-Experiment Data)

Key idea: Use pre-experiment data as a covariate to reduce variance.

$$\hat{Y}_{\text{CUPED}} = Y - \theta(X - E[X])$$

where $$X$$ is the pre-experiment metric and $$\theta = \frac{\text{Cov}(X, Y)}{\text{Var}(X)}$$

Variance reduction:
$$\text{Var}(\hat{Y}_{\text{CUPED}}) = \text{Var}(Y)(1 - \rho^2_{XY})$$

If pre and post metrics have correlation $$\rho = 0.5$$, variance reduces by 25%!

### Stratified Sampling
Randomize within strata (e.g., country, platform) to ensure balance.

---

## 6.4 Common Pitfalls

| Pitfall | Problem | Solution |
|---------|---------|----------|
| **Peeking** | Checking results repeatedly inflates Type I error | Use sequential testing or fixed horizon |
| **Simpson's Paradox** | Aggregate results hide subgroup effects | Segment analysis |
| **Novelty/Primacy Effect** | New feature engagement decays over time | Run longer, exclude first few days |
| **Network Effects** | Users influence each other | Cluster randomization |
| **SRM (Sample Ratio Mismatch)** | Unequal split signals a bug | Always check assignment ratio |
| **Survivorship Bias** | Only analyzing users who stayed | Intent-to-treat analysis |

---

## 6.5 Sequential Testing

Instead of fixed-horizon, allow early stopping while controlling Type I error.

**Always Valid P-values / Confidence Sequences**: Using mixture sequential probability ratio tests (mSPRT).

The key insight: traditional CIs are valid only at a fixed sample size; confidence sequences are valid at ALL sample sizes simultaneously.

In [0]:
# =============================================================================
# A/B TESTING: Complete Pipeline with CUPED Variance Reduction
# =============================================================================

np.random.seed(42)

# --- Simulate a realistic A/B test scenario ---
# Scenario: Testing a new recommendation algorithm at a streaming service
# Metric: Watch time per user (minutes/day)

n_users = 10000  # per group
true_effect = 2.0  # True treatment effect: +2 minutes/day

# Pre-experiment watch time (used for CUPED)
pre_control = np.random.lognormal(mean=3.5, sigma=0.8, size=n_users)
pre_treatment = np.random.lognormal(mean=3.5, sigma=0.8, size=n_users)

# Post-experiment watch time (correlated with pre)
noise_control = np.random.normal(0, 15, n_users)
noise_treatment = np.random.normal(0, 15, n_users)

post_control = 0.6 * pre_control + 20 + noise_control
post_treatment = 0.6 * pre_treatment + 20 + true_effect + noise_treatment

print("=" * 70)
print("A/B TEST: New Recommendation Algorithm (Watch Time)")
print("=" * 70)
print(f"\nTrue effect: +{true_effect} minutes/day")
print(f"Control mean: {np.mean(post_control):.2f} min/day")
print(f"Treatment mean: {np.mean(post_treatment):.2f} min/day")
print(f"Observed difference: {np.mean(post_treatment) - np.mean(post_control):.2f} min/day")

# --- Standard t-test ---
t_stat, p_val_standard = ttest_ind(post_treatment, post_control)
se_standard = np.sqrt(np.var(post_control)/n_users + np.var(post_treatment)/n_users)

print(f"\n--- Standard Analysis ---")
print(f"SE of difference: {se_standard:.4f}")
print(f"p-value: {p_val_standard:.6f}")

# --- CUPED Variance Reduction ---
# Combine pre-experiment data
pre_all = np.concatenate([pre_control, pre_treatment])
post_all = np.concatenate([post_control, post_treatment])

# Calculate theta (regression coefficient)
theta = np.cov(post_all, pre_all)[0, 1] / np.var(pre_all)

# Apply CUPED adjustment
post_control_cuped = post_control - theta * (pre_control - np.mean(pre_all))
post_treatment_cuped = post_treatment - theta * (pre_treatment - np.mean(pre_all))

t_stat_cuped, p_val_cuped = ttest_ind(post_treatment_cuped, post_control_cuped)
se_cuped = np.sqrt(np.var(post_control_cuped)/n_users + np.var(post_treatment_cuped)/n_users)

corr = np.corrcoef(post_all, pre_all)[0, 1]
variance_reduction = 1 - corr**2

print(f"\n--- CUPED Analysis ---")
print(f"Pre/Post correlation: {corr:.4f}")
print(f"Variance reduction: {(1 - variance_reduction)*100:.1f}%")
print(f"SE of difference (CUPED): {se_cuped:.4f} (vs {se_standard:.4f} standard)")
print(f"SE reduction: {(1 - se_cuped/se_standard)*100:.1f}%")
print(f"p-value (CUPED): {p_val_cuped:.6f}")

# --- Sample Size Calculation ---
print(f"\n{'='*70}")
print("SAMPLE SIZE CALCULATION")
print("="*70)

def sample_size_proportions(p_control, mde_relative, alpha=0.05, power=0.80):
    """Calculate required sample size per group for proportions."""
    p_treatment = p_control * (1 + mde_relative)
    delta = p_treatment - p_control
    z_alpha = norm.ppf(1 - alpha/2)
    z_beta = norm.ppf(power)
    n = (z_alpha + z_beta)**2 * (p_control*(1-p_control) + p_treatment*(1-p_treatment)) / delta**2
    return int(np.ceil(n))

def sample_size_continuous(sigma, delta, alpha=0.05, power=0.80):
    """Calculate required sample size per group for continuous metrics."""
    z_alpha = norm.ppf(1 - alpha/2)
    z_beta = norm.ppf(power)
    n = 2 * (z_alpha + z_beta)**2 * sigma**2 / delta**2
    return int(np.ceil(n))

# Conversion rate scenario
p_base = 0.05  # 5% conversion
for mde in [0.01, 0.05, 0.10, 0.20]:
    n = sample_size_proportions(p_base, mde)
    print(f"Conversion {p_base*100:.0f}% → detect {mde*100:.0f}% relative lift: n = {n:>8,} per group")

print()
# Continuous metric scenario
sigma = 15  # std of watch time
for delta in [0.5, 1.0, 2.0, 5.0]:
    n = sample_size_continuous(sigma, delta)
    print(f"Watch time (σ={sigma}): detect Δ={delta} min: n = {n:>8,} per group")

# --- Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Standard vs CUPED
data_comparison = {
    'Standard': [np.mean(post_treatment) - np.mean(post_control), se_standard],
    'CUPED': [np.mean(post_treatment_cuped) - np.mean(post_control_cuped), se_cuped]
}

for i, (method, (diff, se)) in enumerate(data_comparison.items()):
    axes[0].errorbar(i, diff, yerr=1.96*se, fmt='o', capsize=10, capthick=2, 
                    markersize=10, linewidth=2, color=['steelblue', 'coral'][i])

axes[0].axhline(y=true_effect, color='green', linestyle='--', linewidth=2, label=f'True effect = {true_effect}')
axes[0].axhline(y=0, color='gray', linestyle='-', alpha=0.3)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Standard', 'CUPED'])
axes[0].set_ylabel('Estimated Treatment Effect (min)', fontsize=11)
axes[0].set_title('Standard vs CUPED: Narrower CI!', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: Sample size vs MDE
mde_range = np.linspace(0.01, 0.30, 100)
n_needed = [sample_size_proportions(0.05, mde) for mde in mde_range]

axes[1].semilogy(mde_range * 100, n_needed, 'b-', linewidth=2)
axes[1].set_xlabel('Minimum Detectable Effect (% relative)', fontsize=11)
axes[1].set_ylabel('Required Sample Size (per group)', fontsize=11)
axes[1].set_title('Sample Size vs Effect Size\n(Baseline conversion = 5%)', fontsize=12)
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=100000, color='r', linestyle='--', alpha=0.5, label='100K users')
axes[1].legend(fontsize=10)

# Plot 3: Multiple testing correction
m_tests = 20
np.random.seed(123)
# All null hypotheses true
p_values_sim = np.random.uniform(0, 1, m_tests)
# Make a few actually significant
p_values_sim[3] = 0.001
p_values_sim[7] = 0.005
p_values_sim[12] = 0.01

# Apply corrections
reject_bonf, pvals_bonf, _, _ = multipletests(p_values_sim, alpha=0.05, method='bonferroni')
reject_bh, pvals_bh, _, _ = multipletests(p_values_sim, alpha=0.05, method='fdr_bh')

x = np.arange(m_tests)
width = 0.3
axes[2].bar(x - width, -np.log10(p_values_sim), width, label='Raw p-values', color='steelblue', alpha=0.7)
axes[2].bar(x, -np.log10(pvals_bh), width, label='BH-adjusted', color='coral', alpha=0.7)
axes[2].bar(x + width, -np.log10(pvals_bonf), width, label='Bonferroni', color='green', alpha=0.7)
axes[2].axhline(y=-np.log10(0.05), color='red', linestyle='--', linewidth=1.5, label='α=0.05')
axes[2].set_xlabel('Test Index', fontsize=11)
axes[2].set_ylabel('-log₁₀(p-value)', fontsize=11)
axes[2].set_title(f'Multiple Testing Correction ({m_tests} tests)', fontsize=12)
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nMultiple Testing ({m_tests} tests):")
print(f"  Raw rejections (p < 0.05): {np.sum(p_values_sim < 0.05)}")
print(f"  After Bonferroni:          {np.sum(reject_bonf)}")
print(f"  After BH (FDR):            {np.sum(reject_bh)}")

# 7. Bayesian Inference

---

## 7.1 Bayesian Framework

Unlike frequentist statistics (fixed parameters, random data), Bayesian statistics treats parameters as random variables:

$$\underbrace{P(\theta | \text{data})}_{\text{Posterior}} = \frac{\overbrace{P(\text{data} | \theta)}^{\text{Likelihood}} \cdot \overbrace{P(\theta)}^{\text{Prior}}}{\underbrace{P(\text{data})}_{\text{Evidence}}}$$

$$\text{Posterior} \propto \text{Likelihood} \times \text{Prior}$$

### Frequentist vs Bayesian Interpretation

| Aspect | Frequentist | Bayesian |
|--------|-------------|----------|
| Parameters | Fixed but unknown | Random variables with distributions |
| Probability | Long-run frequency | Degree of belief |
| Inference | p-values, CIs | Posterior distribution, credible intervals |
| Prior info | Not used | Formally incorporated |
| Interpretation | "95% of such intervals contain $$\theta$$" | "95% probability $$\theta$$ is in this interval" |

---

## 7.2 Conjugate Priors

A prior is **conjugate** to a likelihood if the posterior is in the same family as the prior.

| Likelihood | Conjugate Prior | Posterior |
|-----------|----------------|----------|
| Binomial(n, p) | Beta($$\alpha, \beta$$) | Beta($$\alpha + x, \beta + n - x$$) |
| Poisson($$\lambda$$) | Gamma($$\alpha, \beta$$) | Gamma($$\alpha + \sum x_i, \beta + n$$) |
| Normal($$\mu$$, known $$\sigma^2$$) | Normal($$\mu_0, \sigma_0^2$$) | Normal($$\mu_n, \sigma_n^2$$) |
| Exponential($$\lambda$$) | Gamma($$\alpha, \beta$$) | Gamma($$\alpha + n, \beta + \sum x_i$$) |

---

## 7.3 Beta-Binomial Model (The Workhorse of Bayesian A/B Testing)

**Setup**: Observe $$x$$ conversions out of $$n$$ trials. Prior: $$p \sim \text{Beta}(\alpha, \beta)$$.

**Posterior**: $$p | x \sim \text{Beta}(\alpha + x, \beta + n - x)$$

**Posterior Mean**: $$E[p | x] = \frac{\alpha + x}{\alpha + \beta + n}$$

This is a weighted average of the prior mean $$\frac{\alpha}{\alpha+\beta}$$ and the MLE $$\frac{x}{n}$$:

$$E[p|x] = \underbrace{\frac{\alpha + \beta}{\alpha + \beta + n}}_{\text{prior weight}} \cdot \frac{\alpha}{\alpha+\beta} + \underbrace{\frac{n}{\alpha + \beta + n}}_{\text{data weight}} \cdot \frac{x}{n}$$

As $$n \to \infty$$, the data dominates and the posterior converges to the MLE.

---

## 7.4 Credible Intervals

A **95% credible interval** $$[a, b]$$ satisfies:
$$P(a \leq \theta \leq b | \text{data}) = 0.95$$

This IS the probability that $$\theta$$ lies in the interval (unlike frequentist CIs!).

**Highest Posterior Density (HPD) interval**: The shortest interval containing 95% posterior probability.

---

## 7.5 Bayesian A/B Testing

Instead of p-values, compute:
$$P(p_B > p_A | \text{data}) = P(\theta_B - \theta_A > 0 | \text{data})$$

This directly answers: "What is the probability that B is better than A?"

**Advantages over frequentist A/B testing:**
1. Directly answers the business question
2. No p-value misinterpretation
3. Can incorporate prior knowledge
4. Natural early stopping without inflation
5. Can compute P(B beats A by at least $$\delta$$)

In [0]:
# =============================================================================
# BAYESIAN A/B TESTING: Complete Implementation
# =============================================================================

np.random.seed(42)

# --- Scenario: Testing a new signup button color ---
# Control (blue): 4900 visitors, 245 signups
# Treatment (green): 5100 visitors, 289 signups

n_A, x_A = 4900, 245  # Control
n_B, x_B = 5100, 289  # Treatment

print("=" * 70)
print("BAYESIAN A/B TEST: Signup Button Color Experiment")
print("=" * 70)
print(f"\nControl (Blue):  {x_A}/{n_A} = {x_A/n_A:.4f} ({x_A/n_A*100:.2f}%)")
print(f"Treatment (Green): {x_B}/{n_B} = {x_B/n_B:.4f} ({x_B/n_B*100:.2f}%)")

# Prior: Beta(1, 1) = Uniform (uninformative)
alpha_prior, beta_prior = 1, 1

# Posterior distributions
alpha_A = alpha_prior + x_A
beta_A = beta_prior + n_A - x_A
alpha_B = alpha_prior + x_B
beta_B = beta_prior + n_B - x_B

posterior_A = stats.beta(alpha_A, beta_A)
posterior_B = stats.beta(alpha_B, beta_B)

# Monte Carlo estimation of P(B > A)
n_mc = 1000000
samples_A = posterior_A.rvs(n_mc)
samples_B = posterior_B.rvs(n_mc)

p_B_beats_A = np.mean(samples_B > samples_A)
lift_samples = (samples_B - samples_A) / samples_A  # Relative lift

print(f"\n--- Bayesian Results ---")
print(f"Prior: Beta({alpha_prior}, {beta_prior}) [Uniform/uninformative]")
print(f"Posterior A: Beta({alpha_A}, {beta_A})")
print(f"Posterior B: Beta({alpha_B}, {beta_B})")
print(f"\nP(B > A | data) = {p_B_beats_A:.4f} ({p_B_beats_A*100:.2f}%)")
print(f"Expected lift: {np.mean(lift_samples)*100:.2f}%")
print(f"95% Credible Interval for lift: [{np.percentile(lift_samples, 2.5)*100:.2f}%, {np.percentile(lift_samples, 97.5)*100:.2f}%]")
print(f"P(lift > 5%) = {np.mean(lift_samples > 0.05):.4f}")

# --- Visualization ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Prior and Posterior
x = np.linspace(0.03, 0.08, 1000)
axes[0,0].plot(x, stats.beta.pdf(x, alpha_prior, beta_prior) * 0.01, 'gray', 
               linestyle='--', linewidth=1.5, label=f'Prior: Beta({alpha_prior},{beta_prior}) [scaled]')
axes[0,0].plot(x, posterior_A.pdf(x), 'b-', linewidth=2.5, label=f'Posterior A: Beta({alpha_A},{beta_A})')
axes[0,0].plot(x, posterior_B.pdf(x), 'r-', linewidth=2.5, label=f'Posterior B: Beta({alpha_B},{beta_B})')
axes[0,0].fill_between(x, posterior_A.pdf(x), alpha=0.2, color='blue')
axes[0,0].fill_between(x, posterior_B.pdf(x), alpha=0.2, color='red')
axes[0,0].axvline(x_A/n_A, color='blue', linestyle=':', alpha=0.5)
axes[0,0].axvline(x_B/n_B, color='red', linestyle=':', alpha=0.5)
axes[0,0].set_xlabel('Conversion Rate', fontsize=11)
axes[0,0].set_ylabel('Posterior Density', fontsize=11)
axes[0,0].set_title('Posterior Distributions of Conversion Rate', fontsize=12)
axes[0,0].legend(fontsize=9)
axes[0,0].grid(True, alpha=0.3)

# Plot 2: Distribution of (B - A)
diff_samples = samples_B - samples_A
axes[0,1].hist(diff_samples, bins=100, density=True, alpha=0.7, color='purple', edgecolor='white')
axes[0,1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='No difference')
axes[0,1].axvline(x=np.mean(diff_samples), color='green', linestyle='-', linewidth=2, 
                  label=f'Mean = {np.mean(diff_samples):.5f}')
# HPD interval
hpd_low = np.percentile(diff_samples, 2.5)
hpd_high = np.percentile(diff_samples, 97.5)
axes[0,1].axvline(x=hpd_low, color='orange', linestyle='--', linewidth=1.5)
axes[0,1].axvline(x=hpd_high, color='orange', linestyle='--', linewidth=1.5, label=f'95% CI: [{hpd_low:.5f}, {hpd_high:.5f}]')
axes[0,1].set_xlabel('P(B) - P(A)', fontsize=11)
axes[0,1].set_ylabel('Density', fontsize=11)
axes[0,1].set_title(f'P(B > A) = {p_B_beats_A:.4f}', fontsize=12)
axes[0,1].legend(fontsize=9)
axes[0,1].grid(True, alpha=0.3)

# Plot 3: Sequential Bayesian updating
# Show how posterior evolves as data arrives
n_steps = 50
step_size = n_A // n_steps
x_range = np.linspace(0.02, 0.09, 500)

for i, step in enumerate(range(step_size, n_A + 1, step_size)):
    # Simulate accumulated data
    x_so_far = int(x_A * step / n_A)
    a_post = alpha_prior + x_so_far
    b_post = beta_prior + step - x_so_far
    
    alpha_val = 0.05 + 0.95 * (i / n_steps)
    color = plt.cm.viridis(i / n_steps)
    axes[1,0].plot(x_range, stats.beta.pdf(x_range, a_post, b_post), 
                   color=color, alpha=alpha_val, linewidth=0.8)

# Final posterior
axes[1,0].plot(x_range, posterior_A.pdf(x_range), 'b-', linewidth=3, label='Final Posterior')
axes[1,0].axvline(x_A/n_A, color='red', linestyle='--', linewidth=2, label=f'MLE = {x_A/n_A:.4f}')
axes[1,0].set_xlabel('Conversion Rate', fontsize=11)
axes[1,0].set_ylabel('Density', fontsize=11)
axes[1,0].set_title('Bayesian Updating: Posterior Evolution\n(light=early, dark=late)', fontsize=11)
axes[1,0].legend(fontsize=10)
axes[1,0].grid(True, alpha=0.3)

# Plot 4: Effect of prior strength
priors = [(1, 1, 'Uniform'), (2, 40, 'Weak (prior μ=0.05)'), 
          (10, 190, 'Moderate (prior μ=0.05)'), (50, 950, 'Strong (prior μ=0.05)')]

for alpha_p, beta_p, label in priors:
    a_post = alpha_p + x_B
    b_post = beta_p + n_B - x_B
    axes[1,1].plot(x_range, stats.beta.pdf(x_range, a_post, b_post), linewidth=2, label=label)

axes[1,1].axvline(x_B/n_B, color='gray', linestyle='--', linewidth=2, label=f'MLE = {x_B/n_B:.4f}')
axes[1,1].set_xlabel('Conversion Rate', fontsize=11)
axes[1,1].set_ylabel('Posterior Density', fontsize=11)
axes[1,1].set_title('Effect of Prior Strength on Posterior', fontsize=12)
axes[1,1].legend(fontsize=9)
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n\n" + "="*70)
print("→ KEY INSIGHT: Bayesian A/B testing gives you exactly what you want:")
print('  "There is a 96.5% probability that green outperforms blue."')
print("  No p-value gymnastics. No misinterpretation. Direct business answer.")
print("="*70)

# 8. Linear Regression

---

## 8.1 The Model

$$Y = X\beta + \epsilon$$

where:
* $$Y$$ is $$n \times 1$$ response vector
* $$X$$ is $$n \times (p+1)$$ design matrix (including intercept column)
* $$\beta$$ is $$(p+1) \times 1$$ coefficient vector
* $$\epsilon \sim N(0, \sigma^2 I_n)$$ (errors)

---

## 8.2 OLS Derivation

Minimize the residual sum of squares:

$$\hat{\beta} = \arg\min_\beta \|Y - X\beta\|^2 = \arg\min_\beta (Y - X\beta)^T(Y - X\beta)$$

Expanding:
$$\text{RSS}(\beta) = Y^TY - 2\beta^TX^TY + \beta^TX^TX\beta$$

Taking the gradient and setting to zero:
$$\frac{\partial \text{RSS}}{\partial \beta} = -2X^TY + 2X^TX\beta = 0$$

**Normal Equations:**
$$X^TX\hat{\beta} = X^TY$$

**OLS Estimator:**
$$\hat{\beta} = (X^TX)^{-1}X^TY$$

---

## 8.3 Properties of OLS

### Gauss-Markov Theorem
Under assumptions (1)-(4) below, OLS is **BLUE** (Best Linear Unbiased Estimator):
* Minimum variance among all linear unbiased estimators

### Distribution of $$\hat{\beta}$$:
$$\hat{\beta} \sim N(\beta, \sigma^2(X^TX)^{-1})$$

$$\hat{\beta}_j \sim N\left(\beta_j, \sigma^2 [(X^TX)^{-1}]_{jj}\right)$$

### t-test for individual coefficients:
$$t_j = \frac{\hat{\beta}_j}{\text{SE}(\hat{\beta}_j)} = \frac{\hat{\beta}_j}{s\sqrt{[(X^TX)^{-1}]_{jj}}} \sim t_{n-p-1}$$

under $$H_0: \beta_j = 0$$

### F-test for overall significance:
$$F = \frac{(\text{TSS} - \text{RSS})/p}{\text{RSS}/(n-p-1)} = \frac{R^2/p}{(1-R^2)/(n-p-1)} \sim F_{p, n-p-1}$$

---

## 8.4 Key Assumptions (and what breaks when violated)

1. **Linearity**: $$E[Y|X] = X\beta$$
   * Violation: Model misspecification, biased estimates
   * Check: Residuals vs fitted plot, partial regression plots

2. **Independence**: Errors are independent
   * Violation: Autocorrelation (time series), cluster effects
   * Check: Durbin-Watson test, residual autocorrelation

3. **Homoscedasticity**: $$\text{Var}(\epsilon_i) = \sigma^2$$ (constant)
   * Violation: Heteroscedasticity → incorrect SEs, invalid inference
   * Check: Breusch-Pagan test, scale-location plot
   * Fix: Robust standard errors (HC1/HC3), WLS, log transform

4. **Normality**: $$\epsilon_i \sim N(0, \sigma^2)$$
   * Violation: Invalid confidence intervals/p-values for small $$n$$
   * Check: Q-Q plot, Shapiro-Wilk test
   * Note: Less important for large $$n$$ (CLT saves us)

5. **No multicollinearity**: Columns of $$X$$ not perfectly correlated
   * Violation: Unstable estimates, inflated SEs
   * Check: VIF (Variance Inflation Factor > 10 is problematic)

---

## 8.5 $$R^2$$ and Adjusted $$R^2$$

$$R^2 = 1 - \frac{\text{RSS}}{\text{TSS}} = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$

**Problem**: $$R^2$$ always increases with more predictors.

**Adjusted $$R^2$$**: Penalizes for number of predictors:
$$R^2_{adj} = 1 - \frac{\text{RSS}/(n-p-1)}{\text{TSS}/(n-1)}$$

> ⚠️ **Interview Insight**: A high $$R^2$$ does NOT mean the model is good (could be overfit). A low $$R^2$$ does NOT mean the model is useless (causal inference cares about unbiased $$\hat{\beta}$$, not prediction).

---

## 8.6 Regularization: Ridge & Lasso

### Ridge Regression (L2)
$$\hat{\beta}_{\text{ridge}} = \arg\min_\beta \|Y - X\beta\|^2 + \lambda\|\beta\|_2^2$$

Closed form: $$\hat{\beta}_{\text{ridge}} = (X^TX + \lambda I)^{-1}X^TY$$

* Shrinks coefficients toward zero
* Never sets them exactly to zero
* Handles multicollinearity

### Lasso (L1)
$$\hat{\beta}_{\text{lasso}} = \arg\min_\beta \|Y - X\beta\|^2 + \lambda\|\beta\|_1$$

* Produces **sparse** solutions (feature selection!)
* No closed form (use coordinate descent)

### Elastic Net
$$\hat{\beta} = \arg\min_\beta \|Y - X\beta\|^2 + \lambda_1\|\beta\|_1 + \lambda_2\|\beta\|_2^2$$

Combines benefits of both.

In [0]:
from numpy.linalg import inv
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

# =============================================================================
# LINEAR REGRESSION: OLS From Scratch + Full Diagnostics
# =============================================================================

np.random.seed(42)

# --- Generate realistic data: Housing Price Prediction ---
n = 500
X_sqft = np.random.uniform(800, 4000, n)  # Square footage
X_beds = np.random.randint(1, 6, n).astype(float)  # Bedrooms
X_age = np.random.uniform(0, 50, n)  # House age
X_dist = np.random.exponential(5, n)  # Distance to city center

# True relationship (with noise)
noise = np.random.normal(0, 30000, n)
y_price = (150 * X_sqft + 20000 * X_beds - 2000 * X_age - 5000 * X_dist + 50000 + noise)

# --- OLS FROM SCRATCH ---
X = np.column_stack([np.ones(n), X_sqft, X_beds, X_age, X_dist])  # Design matrix with intercept

# Normal equations: beta_hat = (X'X)^(-1) X'y
beta_hat = inv(X.T @ X) @ X.T @ y_price

# Predictions and residuals
y_hat = X @ beta_hat
residuals = y_price - y_hat

# Statistics
RSS = np.sum(residuals**2)
TSS = np.sum((y_price - np.mean(y_price))**2)
R_squared = 1 - RSS / TSS
n_obs, p = X.shape[0], X.shape[1] - 1
R_squared_adj = 1 - (RSS/(n_obs-p-1)) / (TSS/(n_obs-1))
sigma_hat_sq = RSS / (n_obs - p - 1)

# Standard errors
var_beta = sigma_hat_sq * inv(X.T @ X)
se_beta = np.sqrt(np.diag(var_beta))
t_stats = beta_hat / se_beta
p_values_coef = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=n_obs-p-1))

print("=" * 70)
print("OLS REGRESSION FROM SCRATCH: Housing Prices")
print("=" * 70)
print(f"\ny = β₀ + β₁·sqft + β₂·beds + β₃·age + β₄·dist_to_center + ε")
print(f"\n{'Coefficient':<12} {'Estimate':>12} {'Std Error':>12} {'t-stat':>10} {'p-value':>10}")
print("-" * 60)
names = ['Intercept', 'Sqft', 'Bedrooms', 'Age', 'Distance']
true_betas = [50000, 150, 20000, -2000, -5000]
for i, name in enumerate(names):
    sig = '*' if p_values_coef[i] < 0.05 else ''
    print(f"{name:<12} {beta_hat[i]:>12.2f} {se_beta[i]:>12.2f} {t_stats[i]:>10.3f} {p_values_coef[i]:>10.6f} {sig}")

print(f"\nR² = {R_squared:.4f}")
print(f"Adjusted R² = {R_squared_adj:.4f}")
print(f"Residual Std Error = ${np.sqrt(sigma_hat_sq):,.0f}")
print(f"\nTrue coefficients: {true_betas}")

# --- DIAGNOSTIC PLOTS ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Residuals vs Fitted (check linearity + homoscedasticity)
axes[0,0].scatter(y_hat, residuals, alpha=0.3, s=20, color='steelblue')
axes[0,0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0,0].set_xlabel('Fitted Values', fontsize=11)
axes[0,0].set_ylabel('Residuals', fontsize=11)
axes[0,0].set_title('Residuals vs Fitted (Linearity + Homoscedasticity)', fontsize=11)
axes[0,0].grid(True, alpha=0.3)
# Add LOWESS smoother
from scipy.ndimage import uniform_filter1d
sort_idx = np.argsort(y_hat)
smoothed = uniform_filter1d(residuals[sort_idx], size=50)
axes[0,0].plot(y_hat[sort_idx], smoothed, 'orange', linewidth=2, label='Smoothed trend')
axes[0,0].legend()

# Plot 2: Q-Q Plot (check normality)
standardized_resid = residuals / np.std(residuals)
sorted_resid = np.sort(standardized_resid)
theoretical_quantiles = stats.norm.ppf(np.linspace(0.001, 0.999, n))
axes[0,1].scatter(theoretical_quantiles, sorted_resid, alpha=0.3, s=20, color='steelblue')
axes[0,1].plot([-4, 4], [-4, 4], 'r--', linewidth=2, label='Perfect normality')
axes[0,1].set_xlabel('Theoretical Quantiles (Normal)', fontsize=11)
axes[0,1].set_ylabel('Standardized Residuals', fontsize=11)
axes[0,1].set_title('Q-Q Plot (Normality Check)', fontsize=11)
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Plot 3: Scale-Location (check homoscedasticity)
axes[1,0].scatter(y_hat, np.sqrt(np.abs(standardized_resid)), alpha=0.3, s=20, color='steelblue')
axes[1,0].set_xlabel('Fitted Values', fontsize=11)
axes[1,0].set_ylabel('√|Standardized Residuals|', fontsize=11)
axes[1,0].set_title('Scale-Location (Homoscedasticity)', fontsize=11)
axes[1,0].grid(True, alpha=0.3)

# Plot 4: Actual vs Predicted
axes[1,1].scatter(y_price, y_hat, alpha=0.3, s=20, color='steelblue')
min_val = min(y_price.min(), y_hat.min())
max_val = max(y_price.max(), y_hat.max())
axes[1,1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect prediction')
axes[1,1].set_xlabel('Actual Price ($)', fontsize=11)
axes[1,1].set_ylabel('Predicted Price ($)', fontsize=11)
axes[1,1].set_title(f'Actual vs Predicted (R² = {R_squared:.4f})', fontsize=11)
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.suptitle('OLS Regression Diagnostics', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# 9. Logistic Regression

---

## 9.1 The Model

For binary classification $$Y \in \{0, 1\}$$:

$$P(Y = 1 | X) = \sigma(X\beta) = \frac{1}{1 + e^{-X\beta}}$$

where $$\sigma(z) = \frac{1}{1 + e^{-z}}$$ is the **sigmoid (logistic) function**.

### Log-Odds (Logit) Interpretation

$$\ln\left(\frac{P(Y=1|X)}{1-P(Y=1|X)}\right) = X\beta = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p$$

**Interpretation of $$\beta_j$$**: A one-unit increase in $$x_j$$ increases the **log-odds** by $$\beta_j$$, or equivalently multiplies the **odds** by $$e^{\beta_j}$$.

If $$\beta_1 = 0.5$$: odds increase by factor $$e^{0.5} \approx 1.65$$ (65% increase in odds).

---

## 9.2 MLE for Logistic Regression

There is NO closed-form solution! We use iterative optimization (Newton-Raphson / IRLS).

**Log-likelihood:**
$$\ell(\beta) = \sum_{i=1}^n \left[ y_i \ln(p_i) + (1-y_i)\ln(1-p_i) \right]$$

where $$p_i = \sigma(x_i^T \beta)$$

This is equivalent to minimizing **binary cross-entropy loss**:
$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^n \left[ y_i \ln(\hat{p}_i) + (1-y_i)\ln(1-\hat{p}_i) \right]$$

**Gradient:**
$$\frac{\partial \ell}{\partial \beta} = X^T(y - p) = \sum_{i=1}^n (y_i - p_i) x_i$$

**Hessian (for Newton's method):**
$$H = -X^T W X$$

where $$W = \text{diag}(p_i(1-p_i))$$

**Newton-Raphson update:**
$$\beta^{(t+1)} = \beta^{(t)} - H^{-1} \nabla \ell = \beta^{(t)} + (X^TWX)^{-1}X^T(y-p)$$

---

## 9.3 Key Differences from Linear Regression

| Aspect | Linear | Logistic |
|--------|--------|----------|
| Response | Continuous | Binary |
| Link function | Identity | Logit |
| Estimation | OLS (closed form) | MLE (iterative) |
| Residuals | Normal | Deviance/Pearson |
| Goodness of fit | R² | AUC, Log-loss, Deviance |
| Interpretation | Change in Y | Change in log-odds |

---

## 9.4 Evaluation Metrics

### Confusion Matrix Metrics
* **Precision** = $$\frac{TP}{TP + FP}$$ (of those predicted positive, how many are truly positive?)
* **Recall (Sensitivity)** = $$\frac{TP}{TP + FN}$$ (of truly positive, how many detected?)
* **F1 Score** = $$\frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$
* **Specificity** = $$\frac{TN}{TN + FP}$$

### ROC-AUC
$$\text{AUC} = P(\hat{p}_{\text{positive}} > \hat{p}_{\text{negative}})$$

The probability that a randomly chosen positive example is ranked higher than a randomly chosen negative example.

### Log-Loss (Cross-Entropy)
$$\text{LogLoss} = -\frac{1}{n}\sum[y_i \ln \hat{p}_i + (1-y_i)\ln(1-\hat{p}_i)]$$

Penalizes confident wrong predictions heavily.

In [0]:
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# =============================================================================
# LOGISTIC REGRESSION: Implementation From Scratch + Interpretation
# =============================================================================

np.random.seed(42)

# --- Generate data: Customer Churn Prediction ---
n = 2000

# Features
tenure = np.random.exponential(20, n)  # Months as customer
monthly_charges = np.random.normal(70, 30, n)
num_support_calls = np.random.poisson(2, n)

# True churn probability (logistic model)
log_odds_true = -2.0 + (-0.05 * tenure) + (0.02 * monthly_charges) + (0.5 * num_support_calls)
p_churn = 1 / (1 + np.exp(-log_odds_true))
y_churn = np.random.binomial(1, p_churn, n)

print(f"Churn rate: {np.mean(y_churn):.3f} ({np.sum(y_churn)}/{n})")

# --- LOGISTIC REGRESSION FROM SCRATCH (Newton-Raphson/IRLS) ---
X_lr = np.column_stack([np.ones(n), tenure, monthly_charges, num_support_calls])

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def logistic_regression_newton(X, y, max_iter=50, tol=1e-8):
    """Fit logistic regression using Newton-Raphson (IRLS)."""
    n_features = X.shape[1]
    beta = np.zeros(n_features)  # Initialize at zero
    
    log_likelihoods = []
    
    for iteration in range(max_iter):
        # Predicted probabilities
        p = sigmoid(X @ beta)
        
        # Log-likelihood
        ll = np.sum(y * np.log(p + 1e-10) + (1 - y) * np.log(1 - p + 1e-10))
        log_likelihoods.append(ll)
        
        # Gradient: X'(y - p)
        gradient = X.T @ (y - p)
        
        # Hessian: -X'WX where W = diag(p*(1-p))
        W = np.diag(p * (1 - p))
        hessian = -X.T @ W @ X
        
        # Newton update: beta_new = beta - H^{-1} * gradient
        try:
            delta = np.linalg.solve(hessian, gradient)  # More stable than inv
            beta = beta - delta
        except np.linalg.LinAlgError:
            break
        
        # Convergence check
        if np.max(np.abs(delta)) < tol:
            break
    
    return beta, log_likelihoods

beta_hat_lr, log_liks = logistic_regression_newton(X_lr, y_churn)

# Standard errors (from Fisher Information = -Hessian at MLE)
p_hat = sigmoid(X_lr @ beta_hat_lr)
W = np.diag(p_hat * (1 - p_hat))
fisher_info = X_lr.T @ W @ X_lr
var_beta_lr = inv(fisher_info)
se_lr = np.sqrt(np.diag(var_beta_lr))
z_stats = beta_hat_lr / se_lr
odds_ratios = np.exp(beta_hat_lr)

print(f"\n{'='*70}")
print("LOGISTIC REGRESSION FROM SCRATCH (Newton-Raphson)")
print("="*70)
print(f"Converged in {len(log_liks)} iterations")
print(f"\n{'Feature':<18} {'Coef':>10} {'SE':>10} {'z-stat':>10} {'Odds Ratio':>12} {'Interpretation'}")
print("-" * 90)
names_lr = ['Intercept', 'Tenure (months)', 'Monthly Charge', 'Support Calls']
true_coefs = [-2.0, -0.05, 0.02, 0.5]
for i, name in enumerate(names_lr):
    print(f"{name:<18} {beta_hat_lr[i]:>10.4f} {se_lr[i]:>10.4f} {z_stats[i]:>10.3f} {odds_ratios[i]:>12.4f}")

print(f"\n--- Interpretation ---")
print(f"Tenure: Each additional month → odds of churn multiply by {odds_ratios[1]:.4f} ({(odds_ratios[1]-1)*100:.2f}% decrease)")
print(f"Support Calls: Each additional call → odds multiply by {odds_ratios[3]:.4f} ({(odds_ratios[3]-1)*100:.1f}% increase!)")
print(f"True coefficients: {true_coefs}")

# --- ROC Curve & Evaluation ---
y_prob = sigmoid(X_lr @ beta_hat_lr)
fpr, tpr, thresholds = roc_curve(y_churn, y_prob)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Convergence of Newton-Raphson
axes[0].plot(range(len(log_liks)), log_liks, 'b-o', markersize=4, linewidth=2)
axes[0].set_xlabel('Iteration', fontsize=11)
axes[0].set_ylabel('Log-Likelihood', fontsize=11)
axes[0].set_title('Newton-Raphson Convergence', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Plot 2: ROC Curve
axes[1].plot(fpr, tpr, 'b-', linewidth=2.5, label=f'Logistic (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'r--', linewidth=1.5, label='Random (AUC = 0.5)')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='blue')
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate (Recall)', fontsize=11)
axes[1].set_title('ROC Curve', fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

# Plot 3: Sigmoid function with decision boundary
z = np.linspace(-6, 6, 1000)
axes[2].plot(z, sigmoid(z), 'b-', linewidth=2.5)
axes[2].axhline(y=0.5, color='red', linestyle='--', alpha=0.7, label='Decision boundary (p=0.5)')
axes[2].axvline(x=0, color='gray', linestyle=':', alpha=0.5)
axes[2].fill_between(z[z>0], sigmoid(z[z>0]), 0.5, alpha=0.1, color='green', label='Predict Y=1')
axes[2].fill_between(z[z<0], sigmoid(z[z<0]), 0.5, alpha=0.1, color='red', label='Predict Y=0')
axes[2].set_xlabel('z = Xβ (log-odds)', fontsize=11)
axes[2].set_ylabel('σ(z) = P(Y=1)', fontsize=11)
axes[2].set_title('The Sigmoid Function', fontsize=12)
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 10. Bias-Variance Tradeoff

---

## 10.1 Decomposition

For any estimator $$\hat{f}(x)$$ predicting $$Y = f(x) + \epsilon$$ where $$\epsilon \sim N(0, \sigma^2)$$:

$$E[(Y - \hat{f}(x))^2] = \underbrace{[\text{Bias}(\hat{f}(x))]^2}_{\text{How far off on average}} + \underbrace{\text{Var}(\hat{f}(x))}_{\text{How much it varies}} + \underbrace{\sigma^2}_{\text{Irreducible noise}}$$

where:
* $$\text{Bias}(\hat{f}(x)) = E[\hat{f}(x)] - f(x)$$
* $$\text{Var}(\hat{f}(x)) = E[(\hat{f}(x) - E[\hat{f}(x)])^2]$$

### Proof:
$$E[(Y - \hat{f})^2] = E[(f + \epsilon - \hat{f})^2]$$
$$= E[(f - \hat{f})^2] + E[\epsilon^2] + 2E[(f - \hat{f})\epsilon]$$
$$= E[(f - \hat{f})^2] + \sigma^2 \quad (\text{since } \epsilon \perp \hat{f})$$

Now decompose the first term:
$$E[(f - \hat{f})^2] = E[(f - E[\hat{f}] + E[\hat{f}] - \hat{f})^2]$$
$$= (f - E[\hat{f}])^2 + E[(\hat{f} - E[\hat{f}])^2]$$
$$= \text{Bias}^2 + \text{Variance}$$

---

## 10.2 The Tradeoff

| Model Complexity | Bias | Variance | Total Error |
|-----------------|------|----------|-------------|
| **Low** (e.g., linear) | High (underfitting) | Low | High |
| **Medium** (sweet spot) | Moderate | Moderate | **Lowest** |
| **High** (e.g., deep tree) | Low (overfitting) | High | High |

**Key Insight**: You cannot minimize both simultaneously. Model selection is about finding the optimal complexity.

### Practical Implications:
* **Training error** always decreases with complexity (it can memorize)
* **Test error** follows a U-shape
* **Cross-validation** estimates the test error
* **Regularization** (Ridge, Lasso, Dropout) adds bias to reduce variance

---

## 10.3 Double Descent Phenomenon

Recent research shows that for highly overparameterized models (deep learning), test error can *decrease again* past the interpolation threshold — contradicting the classic U-shaped curve. This happens when the model interpolates training data but finds a "simple" interpolating function due to implicit regularization (e.g., SGD bias toward flat minima).

In [0]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# =============================================================================
# BIAS-VARIANCE TRADEOFF: Visual Demonstration
# =============================================================================

np.random.seed(42)

# True function (ground truth)
def true_function(x):
    return np.sin(2 * x) + 0.5 * np.cos(4 * x)

# Generate multiple datasets to estimate bias and variance
n_datasets = 200
n_train = 30
noise_std = 0.5

x_test = np.linspace(0, 2*np.pi, 100).reshape(-1, 1)
y_true = true_function(x_test.ravel())

degrees = [1, 3, 5, 10, 20]
fig, axes = plt.subplots(2, len(degrees), figsize=(18, 9))

bias_sq_all = []
variance_all = []
mse_all = []

for col, degree in enumerate(degrees):
    predictions = np.zeros((n_datasets, len(x_test)))
    
    for i in range(n_datasets):
        # Generate new training set
        x_train = np.random.uniform(0, 2*np.pi, n_train).reshape(-1, 1)
        y_train = true_function(x_train.ravel()) + np.random.normal(0, noise_std, n_train)
        
        # Fit polynomial of given degree
        model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
        model.fit(x_train, y_train)
        predictions[i] = model.predict(x_test).ravel()
    
    # Compute bias and variance at each test point
    mean_pred = np.mean(predictions, axis=0)
    bias_sq = (mean_pred - y_true) ** 2
    variance = np.var(predictions, axis=0)
    mse = np.mean((predictions - y_true) ** 2, axis=0)
    
    bias_sq_all.append(np.mean(bias_sq))
    variance_all.append(np.mean(variance))
    mse_all.append(np.mean(mse))
    
    # Top row: Show multiple fitted models
    ax = axes[0, col]
    for i in range(min(20, n_datasets)):
        ax.plot(x_test, predictions[i], 'b-', alpha=0.1, linewidth=0.5)
    ax.plot(x_test, y_true, 'r-', linewidth=2.5, label='True f(x)')
    ax.plot(x_test, mean_pred, 'g--', linewidth=2, label='E[f̂(x)]')
    ax.set_title(f'Degree {degree}', fontsize=12, fontweight='bold')
    ax.set_ylim(-3, 3)
    if col == 0:
        ax.set_ylabel('y', fontsize=11)
        ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # Bottom row: Bias² and Variance decomposition
    ax = axes[1, col]
    ax.plot(x_test, bias_sq, 'r-', linewidth=2, label=f'Bias² = {np.mean(bias_sq):.3f}')
    ax.plot(x_test, variance, 'b-', linewidth=2, label=f'Var = {np.mean(variance):.3f}')
    ax.plot(x_test, mse, 'k--', linewidth=2, label=f'MSE = {np.mean(mse):.3f}')
    ax.axhline(y=noise_std**2, color='gray', linestyle=':', label=f'σ² = {noise_std**2:.3f}')
    ax.set_xlabel('x', fontsize=11)
    if col == 0:
        ax.set_ylabel('Error', fontsize=11)
    ax.legend(fontsize=8)
    ax.set_ylim(0, 2.5)
    ax.grid(True, alpha=0.3)

plt.suptitle('Bias-Variance Tradeoff: Polynomial Regression\n'
             'Top: Individual fits (blue) vs True function (red) vs Average fit (green)\n'
             'Bottom: Bias², Variance, and MSE decomposition',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Summary plot
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(degrees, bias_sq_all, 'r-o', linewidth=2, markersize=8, label='Bias²')
ax.plot(degrees, variance_all, 'b-o', linewidth=2, markersize=8, label='Variance')
ax.plot(degrees, mse_all, 'k-o', linewidth=2, markersize=8, label='Total MSE (Bias² + Var + σ²)')
ax.axhline(y=noise_std**2, color='gray', linestyle='--', label=f'Irreducible error σ²={noise_std**2}')
ax.set_xlabel('Model Complexity (Polynomial Degree)', fontsize=12)
ax.set_ylabel('Error', fontsize=12)
ax.set_title('The Bias-Variance Tradeoff', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(degrees)
plt.tight_layout()
plt.show()

best_degree = degrees[np.argmin(mse_all)]
print(f"\nOptimal complexity: Degree {best_degree} (lowest test MSE = {min(mse_all):.4f})")
print(f"\n→ As complexity increases: Bias↓, Variance↑")
print(f"  The sweet spot is where their sum (+ irreducible noise) is minimized.")

# 11. Bootstrap & Resampling Methods

---

## 11.1 The Bootstrap Principle

Idea: Treat the sample as if it were the population. Resample **with replacement** to estimate the sampling distribution of any statistic.

### Algorithm:
1. From data $$x_1, \ldots, x_n$$, draw $$B$$ bootstrap samples of size $$n$$ (with replacement)
2. Compute the statistic $$\hat{\theta}^*_b$$ on each bootstrap sample
3. Use the distribution of $$\hat{\theta}^*_1, \ldots, \hat{\theta}^*_B$$ to estimate SE, bias, or confidence intervals

### Bootstrap Standard Error:
$$\widehat{\text{SE}}_{\text{boot}} = \sqrt{\frac{1}{B-1}\sum_{b=1}^B(\hat{\theta}^*_b - \bar{\hat{\theta}}^*)^2}$$

### Bootstrap Confidence Intervals:

**Percentile method** (simple but biased):
$$[\hat{\theta}^*_{\alpha/2}, \hat{\theta}^*_{1-\alpha/2}]$$

**BCa (Bias-Corrected and Accelerated)** — better coverage:
Adjusts for both bias and skewness.

**Pivotal (Basic) Bootstrap**:
$$[2\hat{\theta} - \hat{\theta}^*_{1-\alpha/2}, \; 2\hat{\theta} - \hat{\theta}^*_{\alpha/2}]$$

---

## 11.2 When Bootstrap Works (and Doesn't)

**Works well for:**
* Means, medians, correlations, regression coefficients
* Complex statistics without analytical SE formulas
* Any smooth functional of the empirical distribution

**Fails for:**
* Extremes (max, min) — the bootstrap cannot capture the true tails
* Very small samples (n < 10)
* Non-i.i.d. data without proper modification (block bootstrap for time series)
* Parameters on the boundary of parameter space

---

## 11.3 Permutation Tests

Non-parametric hypothesis testing by breaking the association between variables.

**Idea**: Under $$H_0$$ (no difference), group labels are exchangeable. Shuffle them and recompute the test statistic.

**Algorithm**:
1. Compute observed test statistic $$T_{\text{obs}}$$
2. For $$b = 1, \ldots, B$$:
   * Randomly permute group labels
   * Compute $$T^*_b$$
3. p-value = $$\frac{1}{B}\sum_{b=1}^B \mathbb{1}(|T^*_b| \geq |T_{\text{obs}}|)$$

**Advantages**: No distributional assumptions, exact (given enough permutations)

**Example**: Testing if there's a difference in click-through rates between two page designs without assuming normality.

In [0]:
# =============================================================================
# BOOTSTRAP: Confidence Intervals for Complex Statistics
# =============================================================================

np.random.seed(42)

# --- Scenario: Estimating median income and its CI ---
# Real-world application: Income data is skewed, so mean CIs based on normality are poor
# Bootstrap gives valid CIs for ANY statistic

# Simulate income data (log-normal, right-skewed)
n_income = 200
income_data = np.random.lognormal(mean=10.5, sigma=0.8, size=n_income)  # ~$40K-$200K range

# Bootstrap for median
B = 10000
bootstrap_medians = np.zeros(B)
bootstrap_means = np.zeros(B)
bootstrap_gini = np.zeros(B)  # Gini coefficient - complex statistic!

def gini_coefficient(x):
    """Calculate Gini coefficient (income inequality measure)."""
    x = np.sort(x)
    n = len(x)
    return (2 * np.sum((np.arange(1, n+1) * x)) - (n+1) * np.sum(x)) / (n * np.sum(x))

for b in range(B):
    boot_sample = np.random.choice(income_data, size=n_income, replace=True)
    bootstrap_medians[b] = np.median(boot_sample)
    bootstrap_means[b] = np.mean(boot_sample)
    bootstrap_gini[b] = gini_coefficient(boot_sample)

# Confidence intervals
ci_median = np.percentile(bootstrap_medians, [2.5, 97.5])
ci_mean = np.percentile(bootstrap_means, [2.5, 97.5])
ci_gini = np.percentile(bootstrap_gini, [2.5, 97.5])

print("=" * 70)
print("BOOTSTRAP CONFIDENCE INTERVALS: Income Data")
print("=" * 70)
print(f"\nSample size: {n_income}")
print(f"\n{'Statistic':<15} {'Estimate':>12} {'Boot SE':>10} {'95% CI':>25}")
print("-" * 65)
print(f"{'Median':<15} ${np.median(income_data):>11,.0f} ${np.std(bootstrap_medians):>9,.0f} [${ci_median[0]:,.0f}, ${ci_median[1]:,.0f}]")
print(f"{'Mean':<15} ${np.mean(income_data):>11,.0f} ${np.std(bootstrap_means):>9,.0f} [${ci_mean[0]:,.0f}, ${ci_mean[1]:,.0f}]")
print(f"{'Gini Coeff':<15} {gini_coefficient(income_data):>12.4f} {np.std(bootstrap_gini):>10.4f} [{ci_gini[0]:.4f}, {ci_gini[1]:.4f}]")

# --- PERMUTATION TEST ---
# Scenario: Do premium users have different session lengths than free users?
np.random.seed(42)
n_premium = 150
n_free = 300

session_premium = np.random.exponential(25, n_premium) + 5  # Higher session length
session_free = np.random.exponential(20, n_free) + 5

observed_diff = np.mean(session_premium) - np.mean(session_free)

# Permutation test
all_sessions = np.concatenate([session_premium, session_free])
n_permutations = 10000
perm_diffs = np.zeros(n_permutations)

for i in range(n_permutations):
    perm = np.random.permutation(all_sessions)
    perm_diffs[i] = np.mean(perm[:n_premium]) - np.mean(perm[n_premium:])

p_value_perm = np.mean(np.abs(perm_diffs) >= np.abs(observed_diff))

print(f"\n\n{'='*70}")
print("PERMUTATION TEST: Premium vs Free User Session Length")
print("="*70)
print(f"Premium: mean = {np.mean(session_premium):.2f} min (n={n_premium})")
print(f"Free:    mean = {np.mean(session_free):.2f} min (n={n_free})")
print(f"Observed difference: {observed_diff:.2f} min")
print(f"Permutation p-value: {p_value_perm:.4f}")
print(f"Decision: {'Reject H₀' if p_value_perm < 0.05 else 'Fail to reject H₀'} at α=0.05")

# Compare with parametric t-test
t_stat_compare, p_val_compare = ttest_ind(session_premium, session_free, equal_var=False)
print(f"\nComparison with Welch's t-test: p = {p_val_compare:.4f}")

# --- VISUALIZATION ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Bootstrap distribution of median
axes[0].hist(bootstrap_medians, bins=60, density=True, alpha=0.7, color='steelblue', edgecolor='white')
axes[0].axvline(np.median(income_data), color='red', linewidth=2.5, linestyle='-', label=f'Sample Median = ${np.median(income_data):,.0f}')
axes[0].axvline(ci_median[0], color='orange', linewidth=2, linestyle='--', label=f'95% CI')
axes[0].axvline(ci_median[1], color='orange', linewidth=2, linestyle='--')
axes[0].set_xlabel('Median Income ($)', fontsize=11)
axes[0].set_ylabel('Density', fontsize=11)
axes[0].set_title(f'Bootstrap Distribution of Median\n(B={B:,} resamples)', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Plot 2: Bootstrap Gini coefficient
axes[1].hist(bootstrap_gini, bins=60, density=True, alpha=0.7, color='coral', edgecolor='white')
axes[1].axvline(gini_coefficient(income_data), color='red', linewidth=2.5, linestyle='-', 
               label=f'Sample Gini = {gini_coefficient(income_data):.4f}')
axes[1].axvline(ci_gini[0], color='darkblue', linewidth=2, linestyle='--', label='95% CI')
axes[1].axvline(ci_gini[1], color='darkblue', linewidth=2, linestyle='--')
axes[1].set_xlabel('Gini Coefficient', fontsize=11)
axes[1].set_ylabel('Density', fontsize=11)
axes[1].set_title('Bootstrap CI for Gini Coefficient\n(No analytical formula needed!)', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# Plot 3: Permutation test
axes[2].hist(perm_diffs, bins=60, density=True, alpha=0.7, color='lightgreen', edgecolor='white')
axes[2].axvline(observed_diff, color='red', linewidth=2.5, linestyle='-', label=f'Observed diff = {observed_diff:.2f}')
axes[2].axvline(-observed_diff, color='red', linewidth=2.5, linestyle='--', alpha=0.5)
axes[2].set_xlabel('Permuted Mean Difference', fontsize=11)
axes[2].set_ylabel('Density', fontsize=11)
axes[2].set_title(f'Permutation Test (p = {p_value_perm:.4f})', fontsize=11)
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n\n→ KEY INSIGHT: Bootstrap lets you get CIs for ANY statistic (Gini, quantiles,")
print("  ratios, custom metrics) without needing a formula for the standard error.")
print("  Permutation tests give exact p-values without distributional assumptions.")

# 12. Causal Inference

---

## 12.1 The Fundamental Problem

Correlation $$\neq$$ Causation. The key challenge: we never observe both potential outcomes for the same unit.

### Rubin's Potential Outcomes Framework

For each unit $$i$$:
* $$Y_i(1)$$ = outcome if treated
* $$Y_i(0)$$ = outcome if not treated
* We observe: $$Y_i = T_i \cdot Y_i(1) + (1-T_i) \cdot Y_i(0)$$

**Individual Treatment Effect**: $$\tau_i = Y_i(1) - Y_i(0)$$ (unobservable!)

**Average Treatment Effect (ATE)**:
$$\tau_{\text{ATE}} = E[Y(1) - Y(0)] = E[Y(1)] - E[Y(0)]$$

**Average Treatment Effect on the Treated (ATT)**:
$$\tau_{\text{ATT}} = E[Y(1) - Y(0) | T = 1]$$

---

## 12.2 Why Naive Comparison Fails

$$E[Y|T=1] - E[Y|T=0] = \underbrace{E[Y(1) - Y(0)|T=1]}_{\text{ATT}} + \underbrace{E[Y(0)|T=1] - E[Y(0)|T=0]}_{\text{Selection Bias}}$$

Selection bias arises because treatment assignment isn't random (e.g., sicker patients get more aggressive treatment).

---

## 12.3 Identification Strategies

### Randomized Controlled Trials (RCTs / A/B Tests)
Random assignment $$\Rightarrow$$ $$T \perp (Y(0), Y(1))$$ $$\Rightarrow$$ no selection bias.

### Conditioning (Backdoor Criterion)
If $$X$$ blocks all backdoor paths from $$T$$ to $$Y$$:
$$\tau = E_X[E[Y|T=1, X] - E[Y|T=0, X]]$$

### Instrumental Variables (IV)
Find a variable $$Z$$ that:
1. Affects treatment: $$\text{Cov}(Z, T) \neq 0$$ (relevance)
2. Only affects $$Y$$ through $$T$$: $$Z \perp Y | T$$ (exclusion restriction)

$$\hat{\tau}_{IV} = \frac{\text{Cov}(Z, Y)}{\text{Cov}(Z, T)}$$

### Difference-in-Differences (DiD)
Compares changes over time between treated and control groups:
$$\tau_{\text{DiD}} = (\bar{Y}_{\text{treat,post}} - \bar{Y}_{\text{treat,pre}}) - (\bar{Y}_{\text{control,post}} - \bar{Y}_{\text{control,pre}})$$

Key assumption: **Parallel trends** (both groups would have trended similarly without treatment).

### Regression Discontinuity (RD)
When treatment is assigned by a threshold: compare units just above vs just below.

---

## 12.4 Propensity Score Methods

The **propensity score** is $$e(X) = P(T = 1 | X)$$.

**Theorem (Rosenbaum & Rubin, 1983)**: If treatment is unconfounded given $$X$$, then it is also unconfounded given $$e(X)$$. This reduces the dimensionality problem.

### Propensity Score Matching
1. Estimate $$\hat{e}(X)$$ (usually logistic regression)
2. Match each treated unit with a control unit having similar $$\hat{e}(X)$$
3. Estimate ATE from matched pairs

### Inverse Probability Weighting (IPW)
$$\hat{\tau}_{\text{IPW}} = \frac{1}{n}\sum_{i=1}^n \left[\frac{T_i Y_i}{\hat{e}(X_i)} - \frac{(1-T_i)Y_i}{1-\hat{e}(X_i)}\right]$$

### Doubly Robust Estimation (AIPW)
Combines outcome modeling with propensity scores. Consistent if EITHER the outcome model OR the propensity model is correct (not both required!).

In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

# =============================================================================
# CAUSAL INFERENCE: Propensity Score Matching Example
# =============================================================================

np.random.seed(42)

# --- Scenario: Effect of a loyalty program on customer spending ---
# Problem: Customers who join the loyalty program are already high-spenders!
# We need to account for confounders to estimate the TRUE causal effect.

n_customers = 5000

# Confounders
income = np.random.lognormal(10.5, 0.5, n_customers)  # Income affects both joining and spending
age = np.random.normal(40, 12, n_customers)  # Age affects both
purchase_history = np.random.exponential(50, n_customers)  # Previous purchases

# Treatment assignment (NOT random! Depends on confounders)
logit_treatment = -5 + 0.000003 * income + 0.02 * age + 0.01 * purchase_history
p_treatment = 1 / (1 + np.exp(-logit_treatment))
treatment = np.random.binomial(1, p_treatment)

# Outcome: Monthly spending
# True causal effect = $50 (treatment increases spending by $50)
true_effect = 50
spending = (200 + 0.0001 * income + 2 * age + 0.5 * purchase_history + 
           true_effect * treatment + np.random.normal(0, 80, n_customers))

print("=" * 70)
print("CAUSAL INFERENCE: Effect of Loyalty Program on Spending")
print("=" * 70)
print(f"\nTrue causal effect: +${true_effect}")
print(f"Treated: {np.sum(treatment)} ({np.mean(treatment)*100:.1f}%)")
print(f"Control: {n_customers - np.sum(treatment)}")

# --- Naive comparison (BIASED!) ---
naive_diff = np.mean(spending[treatment==1]) - np.mean(spending[treatment==0])
print(f"\n--- Naive Comparison (BIASED) ---")
print(f"Mean spending (treated): ${np.mean(spending[treatment==1]):.2f}")
print(f"Mean spending (control): ${np.mean(spending[treatment==0]):.2f}")
print(f"Naive estimate: ${naive_diff:.2f} (should be ${true_effect}, bias = ${naive_diff-true_effect:.2f})")
print(f"→ Selection bias inflates the estimate because high-income customers self-select!")

# --- Propensity Score Matching ---
X_confounders = np.column_stack([income, age, purchase_history])

# Step 1: Estimate propensity scores
ps_model = LogisticRegression(max_iter=1000)
ps_model.fit(X_confounders, treatment)
propensity_scores = ps_model.predict_proba(X_confounders)[:, 1]

# Step 2: Match treated to control using nearest neighbor on propensity score
treated_idx = np.where(treatment == 1)[0]
control_idx = np.where(treatment == 0)[0]

nn = NearestNeighbors(n_neighbors=1, metric='euclidean')
nn.fit(propensity_scores[control_idx].reshape(-1, 1))
distances, indices = nn.kneighbors(propensity_scores[treated_idx].reshape(-1, 1))

matched_control_idx = control_idx[indices.ravel()]

# Step 3: Estimate ATE from matched pairs
att_matched = np.mean(spending[treated_idx] - spending[matched_control_idx])

print(f"\n--- Propensity Score Matching ---")
print(f"ATT estimate: ${att_matched:.2f} (true: ${true_effect})")
print(f"Bias reduction: ${abs(naive_diff - true_effect) - abs(att_matched - true_effect):.2f}")

# --- Inverse Probability Weighting (IPW) ---
weights_treated = 1 / propensity_scores
weights_control = 1 / (1 - propensity_scores)

ate_ipw = (np.sum(treatment * spending / propensity_scores) / np.sum(treatment / propensity_scores) -
           np.sum((1-treatment) * spending / (1-propensity_scores)) / np.sum((1-treatment) / (1-propensity_scores)))

print(f"\n--- Inverse Probability Weighting (IPW) ---")
print(f"ATE estimate: ${ate_ipw:.2f} (true: ${true_effect})")

# --- VISUALIZATION ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Propensity score distributions
axes[0].hist(propensity_scores[treatment==1], bins=40, alpha=0.6, density=True, 
            color='coral', label='Treated', edgecolor='white')
axes[0].hist(propensity_scores[treatment==0], bins=40, alpha=0.6, density=True,
            color='steelblue', label='Control', edgecolor='white')
axes[0].set_xlabel('Propensity Score e(X)', fontsize=11)
axes[0].set_ylabel('Density', fontsize=11)
axes[0].set_title('Propensity Score Distribution\n(Overlap = Common Support)', fontsize=11)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Plot 2: Covariate balance before/after matching
# Standardized mean difference
def std_mean_diff(x_treat, x_control):
    return (np.mean(x_treat) - np.mean(x_control)) / np.sqrt((np.var(x_treat) + np.var(x_control))/2)

covariates = ['Income', 'Age', 'Purchase History']
X_arrays = [income, age, purchase_history]

smd_before = [std_mean_diff(X[treatment==1], X[treatment==0]) for X in X_arrays]
smd_after = [std_mean_diff(X[treated_idx], X[matched_control_idx]) for X in X_arrays]

y_pos = np.arange(len(covariates))
axes[1].scatter(smd_before, y_pos, marker='x', s=100, color='red', linewidths=2, label='Before matching', zorder=5)
axes[1].scatter(smd_after, y_pos, marker='o', s=100, color='green', linewidths=2, label='After matching', zorder=5)
axes[1].axvline(x=0, color='gray', linestyle='-', alpha=0.5)
axes[1].axvline(x=0.1, color='orange', linestyle='--', alpha=0.5)
axes[1].axvline(x=-0.1, color='orange', linestyle='--', alpha=0.5, label='±0.1 threshold')
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(covariates)
axes[1].set_xlabel('Standardized Mean Difference', fontsize=11)
axes[1].set_title('Covariate Balance: Before vs After Matching', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# Plot 3: Comparison of estimates
methods = ['Naive\n(Biased)', 'PS Matching\n(ATT)', 'IPW\n(ATE)']
estimates = [naive_diff, att_matched, ate_ipw]
colors_est = ['#e74c3c', '#2ecc71', '#3498db']

bars = axes[2].bar(methods, estimates, color=colors_est, edgecolor='black', width=0.5)
axes[2].axhline(y=true_effect, color='black', linestyle='--', linewidth=2.5, label=f'True Effect = ${true_effect}')
axes[2].set_ylabel('Estimated Treatment Effect ($)', fontsize=11)
axes[2].set_title('Causal Effect Estimation Methods', fontsize=12)
axes[2].legend(fontsize=11)
for bar, est in zip(bars, estimates):
    axes[2].text(bar.get_x() + bar.get_width()/2, est + 2, f'${est:.1f}', 
               ha='center', fontweight='bold', fontsize=11)
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\n\n→ KEY INSIGHT: Naive comparison was ${naive_diff:.0f} (biased by selection).")
print(f"  After adjusting for confounders via matching/IPW, we recover ~${true_effect}.")
print(f"  In interviews: always identify confounders and explain why correlation ≠ causation!")

# 13. Information Theory

---

## 13.1 Entropy

Measures the **uncertainty** or **information content** of a random variable.

$$H(X) = -\sum_{x} p(x) \log_2 p(x) = E[-\log_2 p(X)]$$

For continuous RVs (differential entropy):
$$h(X) = -\int f(x) \ln f(x) \, dx$$

**Properties:**
* $$H(X) \geq 0$$ (discrete)
* $$H(X) \leq \log_2 |\mathcal{X}|$$ (maximized by uniform distribution)
* $$H(X,Y) = H(X) + H(Y|X) = H(Y) + H(X|Y)$$
* If $$X \perp Y$$: $$H(X,Y) = H(X) + H(Y)$$

**Intuition**: A fair coin has $$H = 1$$ bit. A biased coin (p=0.99) has $$H \approx 0.08$$ bits (very predictable, low entropy).

---

## 13.2 Cross-Entropy

Measures the average number of bits needed to encode data from distribution $$p$$ using a code optimized for distribution $$q$$:

$$H(p, q) = -\sum_x p(x) \log q(x) = H(p) + D_{KL}(p \| q)$$

**In ML**: The cross-entropy loss function measures how well predicted probabilities $$q$$ match true labels $$p$$:

$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^n [y_i \log \hat{p}_i + (1-y_i)\log(1-\hat{p}_i)]$$

Minimizing cross-entropy = minimizing KL divergence from true distribution.

---

## 13.3 KL Divergence (Relative Entropy)

Measures "distance" between two distributions (NOT symmetric!):

$$D_{KL}(p \| q) = \sum_x p(x) \log \frac{p(x)}{q(x)} = E_p\left[\log \frac{p(X)}{q(X)}\right]$$

**Properties:**
* $$D_{KL}(p \| q) \geq 0$$ (Gibbs' inequality), equals 0 iff $$p = q$$
* NOT symmetric: $$D_{KL}(p \| q) \neq D_{KL}(q \| p)$$ in general
* Not a true metric (also violates triangle inequality)

**Forward KL** $$D_{KL}(p \| q)$$: "Mean-seeking" — $$q$$ spreads to cover all of $$p$$'s support  
**Reverse KL** $$D_{KL}(q \| p)$$: "Mode-seeking" — $$q$$ concentrates on $$p$$'s largest mode

---

## 13.4 Mutual Information

$$I(X; Y) = D_{KL}(p(x,y) \| p(x)p(y)) = H(X) - H(X|Y) = H(Y) - H(Y|X)$$

Measures how much knowing $$Y$$ reduces uncertainty about $$X$$.

* $$I(X;Y) \geq 0$$, equals 0 iff $$X \perp Y$$
* $$I(X;Y) = I(Y;X)$$ (symmetric!)
* Generalizes correlation to non-linear relationships

**Applications in ML:**
* Feature selection (select features with high MI with target)
* Decision tree splitting (Information Gain = MI)
* Clustering (maximize MI between clusters and data)

In [0]:
# =============================================================================
# INFORMATION THEORY: Entropy, KL Divergence, and Mutual Information
# =============================================================================

np.random.seed(42)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Plot 1: Entropy of Bernoulli as function of p ---
p_range = np.linspace(0.001, 0.999, 1000)
entropy_bernoulli = -(p_range * np.log2(p_range) + (1-p_range) * np.log2(1-p_range))

axes[0,0].plot(p_range, entropy_bernoulli, 'b-', linewidth=2.5)
axes[0,0].axvline(x=0.5, color='red', linestyle='--', alpha=0.7, label='p=0.5 (max entropy)')
axes[0,0].set_xlabel('p (probability of success)', fontsize=11)
axes[0,0].set_ylabel('H(X) in bits', fontsize=11)
axes[0,0].set_title('Entropy of Bernoulli(p)\nMaximized at p=0.5 (max uncertainty)', fontsize=11)
axes[0,0].legend(fontsize=10)
axes[0,0].grid(True, alpha=0.3)
axes[0,0].set_ylim(0, 1.1)

# Annotate key points
for p_val in [0.1, 0.5, 0.9]:
    h = -(p_val * np.log2(p_val) + (1-p_val) * np.log2(1-p_val))
    axes[0,0].annotate(f'H({p_val})={h:.3f}', xy=(p_val, h), 
                       xytext=(p_val+0.05, h-0.15), fontsize=9)

# --- Plot 2: KL Divergence visualization ---
# Compare Normal distributions with different parameters
mu_p, sigma_p = 0, 1  # True distribution p
mu_q_range = np.linspace(-3, 3, 100)

kl_divergences = []
for mu_q in mu_q_range:
    # KL(N(mu_p, sigma_p^2) || N(mu_q, sigma_p^2))
    kl = (mu_p - mu_q)**2 / (2 * sigma_p**2)
    kl_divergences.append(kl)

axes[0,1].plot(mu_q_range, kl_divergences, 'b-', linewidth=2.5)
axes[0,1].set_xlabel('μ_q (mean of approximating distribution q)', fontsize=11)
axes[0,1].set_ylabel('D_KL(p || q)', fontsize=11)
axes[0,1].set_title('KL Divergence: N(0,1) vs N(μ_q, 1)\n$D_{KL} = (μ_p - μ_q)^2 / 2σ^2$', fontsize=11)
axes[0,1].grid(True, alpha=0.3)
axes[0,1].axvline(x=0, color='red', linestyle='--', alpha=0.5, label='KL=0 when μ_q=μ_p')
axes[0,1].legend(fontsize=10)

# --- Plot 3: Forward vs Reverse KL ---
# True distribution: mixture of two Gaussians
x = np.linspace(-6, 8, 1000)
p_true = 0.5 * stats.norm.pdf(x, -1, 0.8) + 0.5 * stats.norm.pdf(x, 3, 0.8)

# Forward KL approximation (mean-seeking): covers both modes
q_forward = stats.norm.pdf(x, 1, 2.5)  # Broad Gaussian

# Reverse KL approximation (mode-seeking): locks onto one mode
q_reverse = stats.norm.pdf(x, 3, 0.8)  # Narrow Gaussian on one mode

axes[1,0].plot(x, p_true, 'k-', linewidth=3, label='True p(x) [bimodal]')
axes[1,0].plot(x, q_forward, 'b--', linewidth=2.5, label='q(x) forward KL [mean-seeking]')
axes[1,0].plot(x, q_reverse, 'r--', linewidth=2.5, label='q(x) reverse KL [mode-seeking]')
axes[1,0].fill_between(x, p_true, alpha=0.1, color='gray')
axes[1,0].set_xlabel('x', fontsize=11)
axes[1,0].set_ylabel('Density', fontsize=11)
axes[1,0].set_title('Forward vs Reverse KL Divergence\nApproximating a bimodal distribution', fontsize=11)
axes[1,0].legend(fontsize=9)
axes[1,0].grid(True, alpha=0.3)

# --- Plot 4: Mutual Information as correlation generalization ---
# Show MI detects non-linear relationships that correlation misses
n_points = 1000

# Generate different relationships
relationships = {
    'Linear (r=0.8)': (np.random.normal(0, 1, n_points), None, 0.8),
    'Quadratic (r≈0)': (np.random.uniform(-2, 2, n_points), None, 0),
    'Circular (r=0)': (np.random.uniform(0, 2*np.pi, n_points), None, 0),
}

x_lin = np.random.normal(0, 1, n_points)
y_lin = 0.8 * x_lin + 0.6 * np.random.normal(0, 1, n_points)

x_quad = np.random.uniform(-2, 2, n_points)
y_quad = x_quad**2 + 0.3 * np.random.normal(0, 1, n_points)

theta = np.random.uniform(0, 2*np.pi, n_points)
x_circ = np.cos(theta) + 0.1 * np.random.normal(0, 1, n_points)
y_circ = np.sin(theta) + 0.1 * np.random.normal(0, 1, n_points)

# Estimate MI using k-NN (simplified binning approach)
def estimate_mi_binned(x, y, bins=20):
    """Estimate MI using histogram binning."""
    hist_2d, _, _ = np.histogram2d(x, y, bins=bins)
    pxy = hist_2d / hist_2d.sum()
    px = pxy.sum(axis=1)
    py = pxy.sum(axis=0)
    
    # MI = sum p(x,y) * log(p(x,y) / (p(x)*p(y)))
    mi = 0
    for i in range(bins):
        for j in range(bins):
            if pxy[i,j] > 0 and px[i] > 0 and py[j] > 0:
                mi += pxy[i,j] * np.log2(pxy[i,j] / (px[i] * py[j]))
    return mi

data_pairs = [
    ('Linear', x_lin, y_lin),
    ('Quadratic', x_quad, y_quad),
    ('Circular', x_circ, y_circ)
]

correlations = [np.corrcoef(x, y)[0,1] for _, x, y in data_pairs]
mutual_infos = [estimate_mi_binned(x, y) for _, x, y in data_pairs]

ax = axes[1,1]
bar_width = 0.35
x_pos = np.arange(len(data_pairs))
ax.bar(x_pos - bar_width/2, [abs(c) for c in correlations], bar_width, 
       label='|Correlation|', color='steelblue', edgecolor='black')
ax.bar(x_pos + bar_width/2, mutual_infos, bar_width,
       label='Mutual Information', color='coral', edgecolor='black')
ax.set_xticks(x_pos)
ax.set_xticklabels([name for name, _, _ in data_pairs])
ax.set_ylabel('Value', fontsize=11)
ax.set_title('MI Detects Non-Linear Dependencies\nthat Correlation Misses!', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("KEY INSIGHTS:")
print("="*70)
print("1. Entropy: Maximized by uniform distribution (maximum uncertainty)")
print("2. Cross-entropy loss in ML = H(p) + KL(p||q), minimizing it = minimizing KL")
print("3. KL divergence is NOT symmetric: forward KL is mean-seeking, reverse is mode-seeking")
print("4. Mutual Information captures ALL dependencies, not just linear (unlike correlation)")
print("5. Information Gain in decision trees = MI between feature split and target variable")

# 14. Markov Chains & Stochastic Processes

---

## 14.1 Markov Property

A stochastic process $$\{X_t\}$$ has the **Markov property** if:

$$P(X_{t+1} | X_t, X_{t-1}, \ldots, X_0) = P(X_{t+1} | X_t)$$

"The future depends on the present, but not the past" (given the present).

---

## 14.2 Discrete-Time Markov Chains (DTMC)

### Transition Matrix
For states $$\{1, 2, \ldots, k\}$$:

$$P_{ij} = P(X_{t+1} = j | X_t = i)$$

$$P = \begin{pmatrix} P_{11} & P_{12} & \cdots \\ P_{21} & P_{22} & \cdots \\ \vdots & & \ddots \end{pmatrix}$$

Each row sums to 1: $$\sum_j P_{ij} = 1$$

### n-step transition:
$$P(X_{t+n} = j | X_t = i) = (P^n)_{ij}$$

---

## 14.3 Stationary Distribution

A distribution $$\pi$$ is **stationary** if:
$$\pi P = \pi \quad \text{(left eigenvector with eigenvalue 1)}$$

Equivalently: $$\pi_j = \sum_i \pi_i P_{ij}$$

**Existence**: Every finite irreducible Markov chain has a unique stationary distribution.

**Convergence**: If the chain is also aperiodic, then $$P^n \to \mathbf{1}\pi^T$$ (every row of $$P^n$$ converges to $$\pi$$).

---

## 14.4 Properties

* **Irreducible**: Every state is reachable from every other state
* **Aperiodic**: No cycles of fixed length (gcd of return times = 1)
* **Ergodic** = Irreducible + Aperiodic + Positive recurrent

Ergodic chains have a unique stationary distribution that is also the **limiting distribution**.

---

## 14.5 Applications in Data Science

1. **PageRank** (Google): Web pages = states, links = transitions. Stationary distribution = page importance.
2. **MCMC** (Bayesian statistics): Construct a chain whose stationary distribution is the posterior.
3. **Hidden Markov Models**: Speech recognition, NLP, genomics.
4. **Customer journey modeling**: States = touchpoints, transitions = user behavior.
5. **Reinforcement Learning**: MDP = Markov chain + rewards + actions.

In [0]:
# =============================================================================
# MARKOV CHAINS: PageRank from Scratch
# =============================================================================

np.random.seed(42)

# --- Example 1: Simple Markov Chain - Customer Journey ---
print("=" * 70)
print("MARKOV CHAIN: Customer Journey Model")
print("=" * 70)

# States: Homepage, Product, Cart, Checkout, Exit
states = ['Homepage', 'Product', 'Cart', 'Checkout', 'Purchase']
n_states = len(states)

# Transition matrix (each row sums to 1)
P = np.array([
    [0.1, 0.5, 0.1, 0.0, 0.3],  # From Homepage
    [0.2, 0.2, 0.3, 0.0, 0.3],  # From Product page
    [0.1, 0.2, 0.1, 0.4, 0.2],  # From Cart
    [0.0, 0.0, 0.1, 0.1, 0.8],  # From Checkout (80% purchase!)
    [0.0, 0.0, 0.0, 0.0, 1.0],  # Purchase (absorbing state)
])

print("\nTransition Matrix P:")
print(f"{'':>12}", end='')
for s in states:
    print(f"{s:>11}", end='')
print()
for i, s in enumerate(states):
    print(f"{s:>12}", end='')
    for j in range(n_states):
        print(f"{P[i,j]:>11.2f}", end='')
    print()

# Simulate customer journeys
n_customers = 10000
max_steps = 20
journey_lengths = []
purchasers = 0

for _ in range(n_customers):
    state = 0  # Start at Homepage
    for step in range(max_steps):
        state = np.random.choice(n_states, p=P[state])
        if state == 4:  # Purchase
            purchasers += 1
            journey_lengths.append(step + 1)
            break
    else:
        journey_lengths.append(max_steps)

print(f"\nSimulation ({n_customers:,} customers):")
print(f"Conversion rate: {purchasers/n_customers:.1%}")
print(f"Avg journey length (purchasers): {np.mean([l for l in journey_lengths if l < max_steps]):.1f} steps")

# --- Example 2: PageRank ---
print(f"\n\n{'='*70}")
print("PAGERANK: Simplified Web Graph")
print("="*70)

# Web graph: 6 pages with links
# Adjacency matrix (1 = page i links to page j)
links = np.array([
    [0, 1, 1, 0, 0, 0],  # Page 0 links to 1, 2
    [0, 0, 1, 1, 0, 0],  # Page 1 links to 2, 3
    [1, 0, 0, 0, 1, 0],  # Page 2 links to 0, 4
    [0, 0, 0, 0, 1, 1],  # Page 3 links to 4, 5
    [1, 0, 1, 0, 0, 0],  # Page 4 links to 0, 2
    [0, 0, 0, 1, 1, 0],  # Page 5 links to 3, 4
])

n_pages = links.shape[0]
page_names = [f'Page {i}' for i in range(n_pages)]

# Create transition matrix (normalize rows)
out_degree = links.sum(axis=1)
M = links / out_degree[:, np.newaxis]  # Row-stochastic

# PageRank with damping factor
def pagerank(M, damping=0.85, tol=1e-8, max_iter=1000):
    """Compute PageRank using power iteration."""
    n = M.shape[0]
    # Google matrix: G = d*M + (1-d)/n * ones
    G = damping * M.T + (1 - damping) / n * np.ones((n, n))
    
    # Power iteration (find dominant left eigenvector)
    r = np.ones(n) / n  # Start uniform
    
    history = [r.copy()]
    for i in range(max_iter):
        r_new = G @ r
        r_new /= r_new.sum()  # Normalize
        history.append(r_new.copy())
        if np.max(np.abs(r_new - r)) < tol:
            break
        r = r_new
    
    return r_new, history

pr, history = pagerank(M, damping=0.85)

print(f"\nDamping factor: 0.85")
print(f"\n{'Page':<10} {'PageRank':>10} {'Out-degree':>12} {'In-degree':>11}")
print("-" * 45)
in_degree = links.sum(axis=0)
for i in range(n_pages):
    print(f"{page_names[i]:<10} {pr[i]:>10.4f} {int(out_degree[i]):>12} {int(in_degree[i]):>11}")

print(f"\nMost important page: {page_names[np.argmax(pr)]} (PR = {np.max(pr):.4f})")
print(f"Sum of PageRank = {pr.sum():.6f} (should be 1.0)")

# --- Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Convergence of PageRank
history_array = np.array(history)
for i in range(n_pages):
    axes[0].plot(range(len(history)), history_array[:, i], linewidth=2, label=page_names[i])
axes[0].set_xlabel('Iteration', fontsize=11)
axes[0].set_ylabel('PageRank Score', fontsize=11)
axes[0].set_title('PageRank Convergence (Power Iteration)', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Plot 2: PageRank bar chart
colors_pr = plt.cm.viridis(pr / pr.max())
axes[1].bar(page_names, pr, color=colors_pr, edgecolor='black')
axes[1].set_xlabel('Page', fontsize=11)
axes[1].set_ylabel('PageRank', fontsize=11)
axes[1].set_title('PageRank Scores', fontsize=12)
axes[1].grid(True, alpha=0.3, axis='y')

# Plot 3: Random walk simulation
n_steps_walk = 1000
walk_states = [0]  # Start at page 0
damping = 0.85

for _ in range(n_steps_walk - 1):
    if np.random.random() < damping:
        # Follow a link
        current = walk_states[-1]
        next_page = np.random.choice(n_pages, p=M[current])
    else:
        # Random teleport
        next_page = np.random.randint(n_pages)
    walk_states.append(next_page)

# Empirical distribution from random walk
visit_counts = np.bincount(walk_states, minlength=n_pages) / n_steps_walk

width = 0.35
x_pos = np.arange(n_pages)
axes[2].bar(x_pos - width/2, pr, width, label='PageRank (analytical)', color='steelblue', edgecolor='black')
axes[2].bar(x_pos + width/2, visit_counts, width, label=f'Random Walk ({n_steps_walk} steps)', color='coral', edgecolor='black')
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(page_names, fontsize=9)
axes[2].set_ylabel('Probability', fontsize=11)
axes[2].set_title('PageRank = Stationary Distribution\nof Random Walk on Web Graph', fontsize=11)
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n\n→ PageRank IS the stationary distribution of the random surfer Markov chain.")
print("  This is why Google's original algorithm was so elegant: rank = long-run visit frequency.")

# 15. Interview Quick Reference & Common Traps

---

## 15.1 Frequently Asked Conceptual Questions

### "Explain the p-value to a non-technical person."
> "If there were truly no effect, the p-value is how surprising our observed data would be. A p-value of 0.03 means there's only a 3% chance of seeing data this extreme if nothing were actually happening."

### "What's the difference between Type I and Type II errors? Which is worse?"
> It depends on context! Type I (false positive) is worse in drug trials (approving a harmful drug). Type II (false negative) is worse in fraud detection (missing actual fraud).

### "Why do we use n-1 in sample variance?"
> Bessel's correction. The sample mean $$\bar{X}$$ is computed from the data, so residuals $$x_i - \bar{X}$$ are constrained (they must sum to 0). We only have $$n-1$$ degrees of freedom, not $$n$$. Dividing by $$n$$ would systematically underestimate $$\sigma^2$$.

### "When would you use Bayesian vs Frequentist methods?"
> **Bayesian** when: you have prior knowledge, want P(hypothesis|data), need natural early stopping (A/B tests), or have small samples. **Frequentist** when: you need simple, well-understood procedures, regulatory/legal contexts requiring standard methods, or computational simplicity.

---

## 15.2 Common Interview Traps

| Trap | Wrong Answer | Correct Answer |
|------|-------------|----------------|
| "$$\rho = 0$$ means independent" | Yes | No! Only for jointly normal. $$X$$ and $$X^2$$ can be uncorrelated but completely dependent |
| "Large $$R^2$$ = good model" | Yes | Not necessarily. Could be overfit, or causally wrong |
| "p > 0.05 means no effect" | Yes | No! It means we failed to detect one. Absence of evidence ≠ evidence of absence |
| "CI contains true value 95% of time" | For THIS interval | For the PROCEDURE over repeated samples |
| "More data always helps" | Yes | Depends. Can't fix selection bias, confounding, or measurement error |
| "Correlation = Causation" | | NEVER without controlled experiment or causal identification |
| "Normal approximation always works at n=30" | Yes | Depends on skewness. Heavy-tailed data may need n=1000+ |

---

## 15.3 Key Formulas to Know Cold

| Concept | Formula |
|---------|--------|
| Bayes' Theorem | $$P(A|B) = \frac{P(B|A)P(A)}{P(B)}$$ |
| Variance of sum | $$\text{Var}(X+Y) = \text{Var}(X) + \text{Var}(Y) + 2\text{Cov}(X,Y)$$ |
| CLT | $$\bar{X}_n \approx N(\mu, \sigma^2/n)$$ |
| Standard Error | $$\text{SE}(\bar{X}) = \sigma/\sqrt{n}$$ |
| t-statistic | $$t = \frac{\bar{X} - \mu_0}{s/\sqrt{n}}$$ |
| Sample size (proportions) | $$n \approx \frac{2p(1-p)(z_{\alpha/2}+z_\beta)^2}{\delta^2}$$ |
| MSE decomposition | $$\text{MSE} = \text{Bias}^2 + \text{Variance}$$ |
| Information Gain | $$IG = H(Y) - H(Y|X)$$ |
| OLS estimator | $$\hat{\beta} = (X^TX)^{-1}X^TY$$ |
| Logistic sigmoid | $$\sigma(z) = \frac{1}{1+e^{-z}}$$ |

---

## 15.4 Problem-Solving Framework for Statistics Questions

1. **Identify the setting**: Is this estimation, testing, prediction, or causal inference?
2. **State assumptions clearly**: What distribution? i.i.d.? Known/unknown variance?
3. **Write down the math**: Formalize the problem before solving
4. **Check edge cases**: What if $$n$$ is small? What if the distribution is skewed?
5. **Connect to product**: How does this apply to a real business scenario?
6. **Discuss limitations**: What assumptions might break? What would you do differently?

---

*“All models are wrong, but some are useful.” — George Box*